# Laya V5.5 — All 28 Layers: Brute-Force Liquid CeNN Replacement

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V55_All28_BruteForce_LiquidCeNN_Colab.ipynb)

## Research question

**Can all 28 ModernBERT FFN output projections (`encoder.layers[0..27].mlp.Wo`) be replaced simultaneously with 1D liquid CeNN recurrences, without collapsing the task quality?**

This notebook **really installs 28 simultaneous CeNN replacements** and independently reports the full 28-layer stress-test. Each candidate FFN replacement uses a 3-tap token-neighbor depthwise convolution, local recurrent updates, and an input-dependent time-step gate. **Attention remains ModernBERT**; no claim is made that all 28 Transformer attention modules or complete layers became CeNN.

### Method (all train and selection decisions are case-disjoint from TEST)

1. Load pinned original teacher `vtava/Laya-1.2-speedup-standalone` and reconstruct the 20 previously selected SVD/pair-delta compressed projections.
2. Build an **identical compressed baseline** using strict state-dict cloning. Verify logits before proceeding.
3. On EVERY layer 0..27, brute-force **WIDTHS=(16,32,64)** × **RECURRENT_STEPS=(1,2)** (168 trials, excluding variants that don't actually reduce that FFN projection's parameter count). For speed, collect real FFN input/output activations from TRAIN and run local MSE distillation for each cell; screen every trial on 24 TRAIN-DISJOINT validation decisions.
4. Choose the best local cell per layer by screen gold accuracy, teacher agreement, JS and saved parameters, never on TEST. Then **force-install all 28 cells concurrently**, measure cumulative degradation layer by layer on the **180-decision validation**, and fine-tune the 28 cells together using gold and teacher-KL on shuffled TRAIN. Retain the best checkpoint by validation correctness (a fidelity score).
5. Evaluate the forced all-28 model once on 200 TEST decisions. Separately rebuild an **accepted subset** from the same saved local cells using a deterministic greedy validation fidelity gate. This is a separate ablation and **not** an all-layer CeNN result. Evaluate it on the same TEST split without using TEST to select any layer.
6. Measure FP32 model storage, CUDA and wall-clock latency versus unchanged compressed baseline, and output a full per-layer/cumulative report.
7. Upload **only an independently validated release candidate** to private Hugging Face `vtava/Laya-v55-all28-liquid-cenn`: by default the forced all-28 model must pass all release gates; never silently substitute a smaller subset under an all28 name. No model weights are downloaded to the browser.
8. Automatically download **statistics-only** `Laya_V55_All28_Statistics.zip`, containing JSON/CSV reports, checksums and Hugging Face status. No `.safetensors` or tokenizer data is included.

Because replacing 28 FFNs with new small cells is much more disruptive than V5.4's one-layer substitution, this is an **aggressive feasibility experiment**, not a promised improvement. The original MODEL source does not change unless this notebook's gated HF publication succeeds. Your original V5.4 model remains untouched.

**Prior result for context:** V5.4 replaced only layer 20's `mlp.Wo` and matched 76.5% teacher gold accuracy on a 200-decision test while reducing FP32 weights ~133 MB. That is not evidence that substituting all 28 will succeed.

[All-28 standalone runtime](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/standalone/laya_v55_all28_liquid/modeling_laya_v55.py)


In [1]:
#@title 1. Pinned requirements, GPU environment and runtime imports
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
import json,math,time,random,hashlib,copy,statistics,gc,importlib.util
from pathlib import Path
from collections import OrderedDict,Counter
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import HfApi,snapshot_download,notebook_login
from safetensors.torch import save_file,load_file
from transformers import AutoTokenizer
assert torch.cuda.is_available(),"Select a Colab GPU (T4 or better)"
DEVICE=torch.device("cuda")
print("Torch:",torch.__version__,"GPU:",torch.cuda.get_device_name(0))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 48.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
Torch: 2.11.0+cu130 GPU: Tesla T4


In [2]:
#@title 2. All28 brute-force configuration and pinned teacher
TEACHER_REPO="vtava/Laya-1.2-speedup-standalone"
PINNED_TEACHER_REVISION="b60184b2976e9f7ffa60c1c42b221d81b3691c3d"
STUDENT_REPO="vtava/Laya-v55-all28-liquid-cenn"
SEED=2047
MAX_LEN=512
TRAIN_CASES=500
DEV_CASES=240
BATCH=2 #@param {type:"integer"}
TRAIN_B=2 #@param {type:"integer"}
TRAIN_MAX_TOKENS=224
SEARCH_WIDTHS=(16,32,64)
SEARCH_STEPS=(1,2)
CE_DT_BOUND=0.50
CAPTURE_BATCHES=3 #@param {type:"integer"}
LOCAL_DISTILL_STEPS=28 #@param {type:"integer"}
LOCAL_DISTILL_LR=0.001 #@param {type:"number"}
SMALL_SCREEN_DECISIONS=24
GOLD_VAL_DECISIONS=180
HISTORICAL_PROBE_DECISIONS=120
AUDIT_DECISIONS=200
FORCE_ALL28_TRAIN_STEPS=140 #@param {type:"integer"}
TRAIN_CHECK_EVERY=20 #@param {type:"integer"}
TRAIN_LR=0.00008 #@param {type:"number"}
TRAIN_TEACHER_KL=0.20
TRAIN_GOLD_CE=0.75
TRAIN_SOFT_GOLD=0.05
GREEDY_MAX_VAL_GOLD_DROP_PP=0.0
GREEDY_MIN_AGREEMENT=0.90
BENCH_REPEATS=10 #@param {type:"integer"}
BASELINE_V52_STANDALONE_DIR="" #@param {type:"string"}
MAX_TEST_TEACHER_GOLD_DROP_PP=1.01
MIN_TEST_AGREEMENT=0.90
MAX_JS=0.008
MIN_SPEED_RATIO=0.90
MIN_TOTAL_FP32_SAVED_BYTES=100_000_000
REQUIRE_ALL_28_FOR_PUBLICATION=True
PUBLISH_TO_HF=True #@param {type:"boolean"}
PRIVATE_HF_REPO=True #@param {type:"boolean"}
ALLOW_UPDATE_EXISTING=False #@param {type:"boolean"}
assert 1<=TRAIN_B<=4 and 1<=BATCH<=8
assert all(4<=w<=128 for w in SEARCH_WIDTHS)
assert all(1<=s<=5 for s in SEARCH_STEPS)
assert 1<=CAPTURE_BATCHES<=16 and 1<=LOCAL_DISTILL_STEPS<=200
assert 1<=TRAIN_CHECK_EVERY<=FORCE_ALL28_TRAIN_STEPS
assert abs(TRAIN_GOLD_CE+TRAIN_SOFT_GOLD+TRAIN_TEACHER_KL-1)<1e-8
DEVICE=torch.device("cuda")
device=DEVICE
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
OUT=Path("/content/Laya_V55_All28_LiquidCeNN")
OUT.mkdir(exist_ok=True,parents=True)
DEST=OUT/"standalone_model"
DEST.mkdir(exist_ok=True,parents=True)
api=HfApi()
try:
    teacher_info=api.model_info(TEACHER_REPO,revision=PINNED_TEACHER_REVISION)
except Exception:
    notebook_login()
    teacher_info=api.model_info(TEACHER_REPO,revision=PINNED_TEACHER_REVISION)
assert teacher_info.sha==PINNED_TEACHER_REVISION
parent_dir=Path(snapshot_download(TEACHER_REPO,revision=PINNED_TEACHER_REVISION))
source_config=json.loads((parent_dir/"config.json").read_text())
assert source_config["format"]=="laya-1p2-full-standalone-v1"
assert int(source_config["head_context_budget"])==64
source_cfg={"head_max_len":192,"max_len":MAX_LEN}
tokenizer=AutoTokenizer.from_pretrained(str(parent_dir/"tokenizer"),local_files_only=True)
print("Pinned V4.6 teacher:",TEACHER_REPO,teacher_info.sha)
print("Training setting:",len(SEARCH_WIDTHS)*len(SEARCH_STEPS)*28,
      "possible CeNN candidate trials across 28 FFNs")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

SOURCE_REVISIONS.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

requirements.txt:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Pinned V4.6 teacher: vtava/Laya-1.2-speedup-standalone b60184b2976e9f7ffa60c1c42b221d81b3691c3d
Training setting: 168 possible CeNN candidate trials across 28 FFNs


In [3]:
#@title 3. Embedded full all28 liquid runtime; pretrained teacher and student
V55_RUNTIME_CODE = "\"\"\"Laya V5.0 all-28-layer brute-force hybrid SVD compression, fully standalone.\n\nRequires only torch, transformers, safetensors. Does NOT download or import\nconvaiinnovations/laya-typed-decisions or TinyCeNN-LM at inference.\nThe model weights and tokenizer must be present in the same export directory.\nOption-score model only; auxiliary action logits are intentionally unavailable.\n\"\"\"\nfrom __future__ import annotations\nimport json\nfrom pathlib import Path\nfrom typing import Any, Mapping, Sequence\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\nfrom safetensors.torch import load_file\nfrom transformers import AutoModel, AutoConfig, AutoTokenizer\n\nQTYPES = {\"choice\": 0, \"score\": 1, \"noul\": 2}\n_DEFAULT_NOUL_LABELS = {\"false\": \"false\", \"true\": \"true\"}\n\n\ndef _serialize_state(state: str | dict | list) -> str:\n    if isinstance(state, str):\n        return state\n    return json.dumps(state, ensure_ascii=False, separators=(\",\", \":\"), default=str)\n\n\ndef _render_value(value: Any) -> str:\n    if isinstance(value, str):\n        return value\n    return json.dumps(value, ensure_ascii=False, separators=(\", \", \": \"), default=str)\n\n\ndef normalize_question(question: Mapping[str, Any]) -> dict[str, Any]:\n    \"\"\"Normalize an external typed-decision question into the runtime format.\"\"\"\n    qtype = str(question.get(\"type\", question.get(\"t\", \"choice\"))).lower()\n    if qtype not in QTYPES:\n        raise ValueError(f\"unsupported question type: {qtype!r}\")\n\n    criteria = question.get(\"criteria\", question.get(\"crit\"))\n    if qtype == \"choice\":\n        if isinstance(criteria, (list, tuple)):\n            criteria = {str(x): None for x in criteria}\n        if not isinstance(criteria, Mapping) or len(criteria) < 2:\n            raise ValueError(\"choice questions require at least two criteria/options\")\n        criteria = dict(criteria)\n    elif qtype == \"score\":\n        if not isinstance(criteria, (list, tuple)) or len(criteria) < 2:\n            raise ValueError(\"score questions require at least two ordered criteria\")\n        criteria = list(criteria)\n    else:\n        if criteria is None:\n            criteria = {}\n        if not isinstance(criteria, Mapping):\n            raise ValueError(\"noul criteria must be a mapping when provided\")\n        criteria = {str(k).lower(): v for k, v in criteria.items()}\n\n    instructions = question.get(\"instructions\", question.get(\"ins\", \"\"))\n    if not isinstance(instructions, str):\n        instructions = json.dumps(instructions, ensure_ascii=False, default=str)\n\n    result = {\"t\": qtype, \"ins\": instructions, \"crit\": criteria}\n    if \"labels\" in question:\n        result[\"labels\"] = dict(question[\"labels\"])\n    return result\n\n\ndef render_options(question: Mapping[str, Any]) -> list[str]:\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    qtype, criteria = q[\"t\"], q.get(\"crit\")\n    if qtype == \"choice\":\n        return [\n            str(k) if v is None or v == \"\" else f\"{k}: {_render_value(v)}\"\n            for k, v in criteria.items()\n        ]\n    if qtype == \"score\":\n        return [f\"level {i}: {_render_value(v)}\" for i, v in enumerate(criteria)]\n\n    labels = q.get(\"labels\", _DEFAULT_NOUL_LABELS)\n    false_label = str(labels.get(\"false\", \"false\")).strip() or \"false\"\n    true_label = str(labels.get(\"true\", \"true\")).strip() or \"true\"\n    false_desc = criteria.get(\"false\")\n    true_desc = criteria.get(\"true\")\n    return [\n        f\"{false_label}: \" + (\n            _render_value(false_desc) if false_desc not in (None, \"\") else \"no, the statement does not hold\"\n        ),\n        f\"{true_label}: \" + (\n            _render_value(true_desc) if true_desc not in (None, \"\") else \"yes, the statement holds\"\n        ),\n    ]\n\n\ndef build_sequence(\n    tokenizer,\n    state: str | dict | list,\n    question: Mapping[str, Any],\n    *,\n    max_len: int = 512,\n    head_max_len: int = 192,\n    truncate_left: bool = False,\n    state_ids: Sequence[int] | None = None,\n) -> tuple[list[int], list[int]]:\n    \"\"\"Build a compact typed-decision sequence and option marker positions.\"\"\"\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    options = render_options(q)\n    mask_token = tokenizer.mask_token\n    instruction = str(q[\"ins\"]).replace(mask_token, \" \")\n    head_ids = tokenizer(\n        f\"{q['t']} question: {instruction}\", add_special_tokens=False\n    )[\"input_ids\"]\n\n    option_ids: list[list[int]] = []\n    for option in options:\n        ids = tokenizer(\n            \" \" + option.replace(mask_token, \" \"),\n            add_special_tokens=False,\n            truncation=True,\n            max_length=48,\n        )[\"input_ids\"]\n        option_ids.append([tokenizer.mask_token_id] + ids)\n\n    option_budget = head_max_len - sum(len(x) for x in option_ids)\n    if option_budget < 16:\n        per = max(4, (head_max_len - 16) // max(1, len(option_ids)))\n        option_ids = [x[:per] for x in option_ids]\n        option_budget = head_max_len - sum(len(x) for x in option_ids)\n    head_ids = head_ids[: max(8, option_budget)]\n\n    ids = [tokenizer.cls_token_id] + head_ids + [tokenizer.sep_token_id]\n    markers: list[int] = []\n    for option in option_ids:\n        markers.append(len(ids))\n        ids.extend(option)\n    ids.append(tokenizer.sep_token_id)\n\n    if state_ids is None:\n        text = _serialize_state(state).replace(mask_token, \" \")\n        state_ids = tokenizer(text, add_special_tokens=False)[\"input_ids\"]\n    room = max(0, max_len - len(ids) - 1)\n    if truncate_left:\n        start = max(0, len(state_ids) - room)\n        state_part = list(state_ids[start:])\n    else:\n        state_part = list(state_ids[:room])\n    ids = (ids + state_part + [tokenizer.sep_token_id])[:max_len]\n    markers = [m for m in markers if m < max_len]\n    return ids, markers\n\n\ndef collate_items(items: Sequence[Mapping[str, Any]], pad_id: int) -> dict[str, Tensor]:\n    if not items:\n        raise ValueError(\"items must not be empty\")\n    n = len(items)\n    length = max(len(x[\"ids\"]) for x in items)\n    kmax = max(len(x[\"markers\"]) for x in items)\n    ids = torch.full((n, length), int(pad_id), dtype=torch.long)\n    mask = torch.zeros((n, length), dtype=torch.long)\n    marker_pos = torch.zeros((n, kmax), dtype=torch.long)\n    marker_mask = torch.zeros((n, kmax), dtype=torch.bool)\n    qtype = torch.zeros(n, dtype=torch.long)\n\n    for i, item in enumerate(items):\n        row = torch.tensor(item[\"ids\"], dtype=torch.long)\n        ids[i, : row.numel()] = row\n        mask[i, : row.numel()] = 1\n        m = torch.tensor(item[\"markers\"], dtype=torch.long)\n        marker_pos[i, : m.numel()] = m\n        marker_mask[i, : m.numel()] = True\n        qtype[i] = int(item[\"qtype\"])\n    return {\n        \"input_ids\": ids,\n        \"attention_mask\": mask,\n        \"marker_pos\": marker_pos,\n        \"marker_mask\": marker_mask,\n        \"qtype\": qtype,\n    }\n\n\n\ndef _apply_rope_compat(cfg):\n    rope = getattr(cfg, \"rope_parameters\", None)\n    if not isinstance(rope,dict):\n        return\n    flat = rope.get(\"rope_theta\")\n    for layer_type, attr in ((\"full_attention\",\"global_rope_theta\"),\n                             (\"sliding_attention\",\"local_rope_theta\")):\n        v = rope.get(layer_type)\n        theta = v.get(\"rope_theta\") if isinstance(v,dict) else flat\n        if theta is not None and hasattr(cfg,attr):\n            setattr(cfg,attr,float(theta))\n\n\nclass LayaStandalone(nn.Module):\n    \"\"\"Native pretrained ModernBERT + exact 64-token option-scoring head.\n\n    ALL weights are stored locally in model.safetensors. No auxiliary act_head.\n    \"\"\"\n    def __init__(self, encoder_config=None, *, budget=64, head_layers=2,\n                 pretrained_components=None):\n        super().__init__()\n        self.budget = int(budget)\n        assert self.budget == 64, \"This checkpoint implements the validated 64-token head\"\n        if pretrained_components is not None:\n            # EXPORT ONLY: reuse the loaded upstream model without duplicating\n            # ~400M weights; this branch is not needed during standalone load.\n            self.encoder = pretrained_components.encoder\n            self.head = pretrained_components.head\n            self.type_emb = pretrained_components.type_emb\n            self.scorer = pretrained_components.scorer\n        else:\n            assert encoder_config is not None\n            _apply_rope_compat(encoder_config)\n            self.encoder = AutoModel.from_config(\n                encoder_config, attn_implementation=\"sdpa\")\n            d = int(encoder_config.hidden_size)\n            nhead = max(1,d//64)\n            layer = nn.TransformerEncoderLayer(\n                d,nhead,dim_feedforward=4*d,dropout=0.1,\n                batch_first=True,norm_first=True)\n            self.head = nn.TransformerEncoder(\n                layer,int(head_layers),enable_nested_tensor=False)\n            self.type_emb = nn.Embedding(3,d)\n            self.scorer = nn.Sequential(nn.LayerNorm(d),nn.Linear(d,d),\n                                        nn.GELU(),nn.Linear(d,1))\n\n    def _select_head_tokens(self,h,attention_mask,marker_pos,marker_mask):\n        # Copied from V4.6, keeping exact order/masks for reproduction.\n        b,t,d=h.shape\n        cls=torch.zeros((b,1),dtype=torch.long,device=h.device)\n        lm_count=min(self.budget,max(t-1,1))\n        lm=torch.linspace(1,max(t-1,1),steps=lm_count,device=h.device).round().long()\n        lm=lm.clamp(max=t-1).view(1,-1).expand(b,-1)\n        marker_ok=marker_mask.bool()\n        lm_ok=attention_mask.bool().gather(1,lm)\n        lm_ok=lm_ok & (lm!=0)\n        duplicate=(lm[:,:,None]==marker_pos[:,None,:]) & marker_ok[:,None,:]\n        lm_ok=lm_ok & ~duplicate.any(-1)\n        idx=torch.cat((cls,marker_pos.clamp(0,t-1),lm),dim=1)\n        keep=torch.cat((torch.ones((b,1),dtype=torch.bool,device=h.device),\n                        marker_ok,lm_ok),dim=1)\n        gathered=h.gather(1,idx[:,:,None].expand(-1,-1,d))\n        return gathered,keep\n\n    def forward(self,input_ids,attention_mask,marker_pos,marker_mask,qtype):\n        h=self.encoder(input_ids=input_ids,\n                       attention_mask=attention_mask).last_hidden_state\n        h=h+self.type_emb(qtype)[:,None,:]\n        compact,keep=self._select_head_tokens(\n            h,attention_mask,marker_pos,marker_mask)\n        transformed=self.head(compact,src_key_padding_mask=~keep)\n        marker_hidden=transformed[:,1:1+marker_pos.shape[1],:]\n        logits=self.scorer(marker_hidden).squeeze(-1).float()\n        logits=logits.masked_fill(~marker_mask,-1e4)\n        return logits,None\n\n    @torch.inference_mode()\n    def predict(self,tokenizer,state,questions,*,max_len=512,head_max_len=192):\n        items=[];names=[];labels=[]\n        for name,raw in questions.items():\n            q=normalize_question(raw)\n            if q[\"t\"]!=\"choice\":\n                raise ValueError(\"Only choice questions supported by predict()\")\n            opts=list(q[\"crit\"].keys())\n            ids,markers=build_sequence(tokenizer,state,q,max_len=max_len,\n                                       head_max_len=head_max_len)\n            if len(markers)!=len(opts):\n                raise ValueError(\"Truncated an option marker\")\n            names.append(name);labels.append(opts)\n            items.append(dict(ids=ids,markers=markers,qtype=QTYPES[q[\"t\"]]))\n        batch=collate_items(items,int(tokenizer.pad_token_id))\n        device=next(self.parameters()).device\n        batch={k:v.to(device) for k,v in batch.items()}\n        with (torch.autocast(\"cuda\",dtype=torch.float16)\n              if device.type==\"cuda\" else __import__(\"contextlib\").nullcontext()):\n            logits,_=self(**batch)\n        probs=logits.float().masked_fill(~batch[\"marker_mask\"],-1e9).softmax(-1).cpu()\n        answers={}\n        for i,name in enumerate(names):\n            p=probs[i,:len(labels[i])]\n            best=int(p.argmax())\n            answers[name]=dict(choice=labels[i][best],\n                               confidence=float(p[best]),\n                               probabilities={label:float(pr) for label,pr in zip(labels[i],p)})\n        return answers\n\n\ndef load_model(model_dir,*,device=\"cpu\",local_files_only=True,compile_head=False):\n    \"\"\"Load entirely from the local standalone directory. No network needed.\"\"\"\n    root=Path(model_dir).resolve()\n    metadata=json.loads((root/\"config.json\").read_text())\n    assert metadata[\"format\"]==\"laya-1p2-full-standalone-v1\"\n    enc_cfg=AutoConfig.from_pretrained(\n        str(root/\"encoder\"),local_files_only=local_files_only)\n    model=LayaStandalone(enc_cfg,budget=int(metadata[\"head_context_budget\"]),\n                         head_layers=int(metadata[\"head_layers\"]))\n    state=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(state,strict=True)\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tok=AutoTokenizer.from_pretrained(\n        str(root/\"tokenizer\"),local_files_only=local_files_only)\n    return model,tok\n\ndef local_pair_indices(model):\n    \"\"\"Verify FULL / same-window LOCAL / LOCAL triples in ModernBERT.\n\n    transformers==4.57.6 uses attn.local_attention: (-1,-1) means FULL,\n    (64,64) is a 128-token sliding window. Newer builds can expose\n    config.layer_types with attn.sliding_window instead. Support both.\n    \"\"\"\n    layers=model.encoder.layers\n    cfg=model.encoder.config\n    n=len(layers)\n    layer_types=getattr(cfg,\"layer_types\",None)\n    if layer_types is not None and len(layer_types)!=n:\n        raise ValueError(\"ModernBERT layer_types count does not match layers\")\n\n    def mode_at(i):\n        a=layers[i].attn\n        old_window=getattr(a,\"local_attention\",None)\n        if old_window is not None:\n            window=tuple(old_window)\n            if window==(-1,-1):\n                return (\"full\",None)\n            if len(window)!=2 or min(window)<0:\n                raise ValueError(f\"Malformed local attention window at {i}: {window}\")\n            return (\"local\",window)\n        typ=(layer_types[i] if layer_types is not None else\n             getattr(layers[i],\"attention_type\",None))\n        window=getattr(a,\"sliding_window\",None)\n        if typ==\"full_attention\" and window is None:\n            return (\"full\",None)\n        if typ==\"sliding_attention\" and window is not None:\n            return (\"local\",int(window))\n        raise ValueError(f\"Unsupported ModernBERT attention metadata at layer {i}: {typ}, {window}\")\n\n    pairs=[]\n    for anchor in range(0,n,3):\n        mode,_=mode_at(anchor)\n        if mode!=\"full\":\n            raise ValueError(f\"Layer {anchor} must be FULL attention\")\n        left_idx,right_idx=anchor+1,anchor+2\n        if right_idx>=n:\n            continue\n        m1,w1=mode_at(left_idx)\n        m2,w2=mode_at(right_idx)\n        if m1!=\"local\" or m2!=\"local\" or w1!=w2:\n            raise ValueError(\n                f\"Layers {(left_idx,right_idx)} must have identical local windows: {m1}/{w1}, {m2}/{w2}\")\n        pairs.append((left_idx,right_idx))\n    return pairs\n\n\nclass SVDDeltaLinear(nn.Module):\n    \"\"\"Second-layer linear projection = shared first-layer projection + A @ B.\n\n    The shared linear is a plain Python reference, NOT a registered submodule;\n    its weights appear only once in state_dict, under the original first layer.\n    \"\"\"\n    def __init__(self, first_linear, rank, *, A=None, B=None):\n        super().__init__()\n        if not isinstance(first_linear,nn.Linear):\n            raise TypeError(f\"Expected nn.Linear, got {type(first_linear)}\")\n        out_dim,in_dim=first_linear.weight.shape\n        rank=int(rank)\n        if not (1<=rank<min(in_dim,out_dim)):\n            raise ValueError(f\"Unsupported rank {rank} for {out_dim}x{in_dim}\")\n        # Using object.__setattr__ prevents double registering the shared module.\n        object.__setattr__(self,\"_shared\",first_linear)\n        self.in_features=int(in_dim)\n        self.out_features=int(out_dim)\n        self.max_rank=rank\n        self.active_rank=rank\n        self.A=nn.Parameter(torch.zeros((out_dim,rank),device=first_linear.weight.device,\n                                        dtype=first_linear.weight.dtype))\n        self.B=nn.Parameter(torch.zeros((rank,in_dim),device=first_linear.weight.device,\n                                        dtype=first_linear.weight.dtype))\n        if A is not None and B is not None:\n            with torch.no_grad():\n                self.A.copy_(A.to(self.A.device,self.A.dtype))\n                self.B.copy_(B.to(self.B.device,self.B.dtype))\n\n    def set_rank(self, rank):\n        rank=int(rank)\n        if rank<1 or rank>self.max_rank:\n            raise ValueError(f\"rank {rank} outside 1..{self.max_rank}\")\n        self.active_rank=rank\n\n    def shrink(self, rank):\n        \"\"\"Permanently discard extra SVD columns for a compact checkpoint.\"\"\"\n        rank=int(rank)\n        if rank<1 or rank>self.max_rank:\n            raise ValueError(rank)\n        self.A=nn.Parameter(self.A[:,:rank].detach().clone())\n        self.B=nn.Parameter(self.B[:rank,:].detach().clone())\n        self.max_rank=rank\n        self.active_rank=rank\n\n    def forward(self,x):\n        # fp16 autocast handles both paths identically, with FP32 stored weights.\n        w=self._shared.weight\n        bias=self._shared.bias\n        base=F.linear(x,w,bias)\n        r=self.active_rank\n        return base+F.linear(F.linear(x,self.B[:r,:]),self.A[:,:r])\n\n\n\n# V4.9 does NOT force compression of all nine local pairs.\n# Each selected follower projection gets a separate low-rank residual;\n# all unselected attention projection matrices retain pretrained weights.\ndef projection_slots(model):\n    slots=[]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            a=getattr(model.encoder.layers[first].attn,component)\n            b=getattr(model.encoder.layers[second].attn,component)\n            if not isinstance(a,nn.Linear) or not isinstance(b,nn.Linear):\n                raise TypeError(f\"Expected nn.Linear for layer {second}.{component}\")\n            if a.weight.shape != b.weight.shape:\n                raise ValueError(f\"Incompatible original projection: {second}.{component}\")\n            if a.bias is not None or b.bias is not None:\n                raise ValueError(f\"Biasful projection needs separate residual bias: {second}.{component}\")\n            slots.append((first,second,component))\n    return slots\n\n\ndef _validate_slot(model, first, second, component):\n    \"\"\"Validate schedule independently of other already-compressed slots.\"\"\"\n    first,second=int(first),int(second)\n    component=str(component)\n    if (first,second) not in local_pair_indices(model):\n        raise ValueError(f\"Unknown local pair: {(first,second)}\")\n    if component not in (\"Wqkv\",\"Wo\"):\n        raise ValueError(f\"Unknown projection: {component}\")\n    return first,second,component\n\n\ndef svd_factors(model, first, second, component, max_rank=256,\n                oversampling=8, niter=2):\n    \"\"\"Compute one factorization per projection from ORIGINAL pretrained weights.\n\n    A and B are float32 CPU tensors and can be sliced for different ranks.\n    \"\"\"\n    first,second,component=_validate_slot(model,first,second,component)\n    a=getattr(model.encoder.layers[first].attn,component)\n    b=getattr(model.encoder.layers[second].attn,component)\n    if not isinstance(a,nn.Linear) or not isinstance(b,nn.Linear):\n        raise ValueError(\"SVD must be done BEFORE that follower is compressed\")\n    with torch.no_grad():\n        delta=b.weight.detach().float()-a.weight.detach().float()\n        r=min(int(max_rank),min(delta.shape)-1)\n        q=min(r+int(oversampling),min(delta.shape))\n        # Randomized SVD (approximation, not an exact mathematical SVD).\n        U,S,V=torch.svd_lowrank(delta,q=q,niter=int(niter))\n        root=S[:r].clamp_min(0).sqrt()\n        A=(U[:,:r]*root.unsqueeze(0)).cpu().contiguous()\n        B=(root.unsqueeze(1)*V[:,:r].T).cpu().contiguous()\n        total=float(delta.square().sum().item())\n        error=float((delta-(A.to(delta.device)@B.to(delta.device))).square().sum().item())\n        del U,S,V,delta\n    return A,B,{\"rank\":r,\n        \"relative_frobenius_error\":(error/max(total,1e-30))**0.5,\n        \"saved_parameter_bytes_if_rank256\":None}\n\n\ndef build_adapter(model, first, second, component, rank, factors=None,\n                  *, initialized=True):\n    \"\"\"Replace ONE follower's projection; return original module for rollback.\n\n    If initialized=False, A/B are zeros until strictly loaded from checkpoint.\n    \"\"\"\n    first=int(first);second=int(second);component=str(component)\n    if (first,second) not in local_pair_indices(model):\n        raise ValueError(f\"Invalid local pair: {(first,second)}\")\n    leader=getattr(model.encoder.layers[first].attn,component)\n    follower_block=model.encoder.layers[second].attn\n    old=getattr(follower_block,component)\n    if not isinstance(leader,nn.Linear) or not isinstance(old,nn.Linear):\n        raise TypeError(\"Projection already wrapped or unavailable\")\n    if leader.bias is not None or old.bias is not None:\n        raise ValueError(\"Biasful attention projections are unsupported\")\n    if initialized:\n        if factors is None:\n            A,B,_=svd_factors(model,first,second,component,max_rank=rank)\n        else:\n            A,B=factors\n        adapter=SVDDeltaLinear(leader,rank,A=A[:,:rank],B=B[:rank,:])\n    else:\n        adapter=SVDDeltaLinear(leader,rank)\n    setattr(follower_block,component,adapter)\n    return old\n\n\ndef restore_original_projection(model, second, component, original):\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Rollback needs the original pretrained linear module\")\n    setattr(model.encoder.layers[int(second)].attn,str(component),original)\n\n\ndef selected_adapters(model):\n    selected=[]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            layer=getattr(model.encoder.layers[second].attn,component)\n            if isinstance(layer,SVDDeltaLinear):\n                selected.append((first,second,component,layer))\n    return selected\n\n\ndef unique_parameter_count(model):\n    return sum(p.numel() for p in model.parameters())\n\n\ndef estimated_saved_bytes(model, first, second, component, rank):\n    leader=getattr(model.encoder.layers[first].attn,component)\n    outdim,indim=leader.weight.shape\n    # Full follower weight removed; A (out×rank), B (rank×in) remain.\n    return 4*(outdim*indim-int(rank)*(outdim+indim))\n\n\ndef load_v49_model(model_dir, *, device=\"cpu\", compile_head=False):\n    \"\"\"Strict, fully offline restoration of selectively compressed model.\"\"\"\n    root=Path(model_dir).resolve()\n    meta=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if meta[\"format\"]!=\"laya-v49-selective-svd-v1\":\n        raise ValueError(\"Wrong model format: expected V4.9 selective SVD\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(meta[\"head_context_budget\"]),\n                         head_layers=int(meta[\"head_layers\"]))\n    for item in meta[\"compressed_projections\"]:\n        build_adapter(model,int(item[\"first\"]),int(item[\"second\"]),\n                      str(item[\"component\"]),int(item[\"rank\"]),\n                      initialized=False)\n    weights=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(weights,strict=True)\n    # No duplicated original follower weights or duplicated leader submodule.\n    assert len(selected_adapters(model))==len(meta[\"compressed_projections\"])\n    for first,second,component,wrapper in selected_adapters(model):\n        leader=getattr(model.encoder.layers[first].attn,component)\n        assert wrapper._shared is leader\n        assert \"_shared\" not in wrapper._modules\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tokenizer=AutoTokenizer.from_pretrained(str(root/\"tokenizer\"),local_files_only=True)\n    return model,tokenizer\n\n\n# ---------------------------------------------------------------------------\n# V5.0: brute-force all 28 ModernBERT blocks, attention + FFN, preserving the\n# proven V4.9 pretrained-pair DELTA path as one selectable candidate family.\n# ---------------------------------------------------------------------------\n\nclass IndependentSVDLinear(nn.Module):\n    \"\"\"Plain original projection compressed as A @ B; original dense W omitted.\n\n    Compatible with ModernBERT's attention Wqkv, Wo and MLP Wi, Wo calls.\n    A saved bias is kept if the original linear was biased.\n    \"\"\"\n    def __init__(self, original: nn.Linear, rank: int, A=None, B=None):\n        super().__init__()\n        if not isinstance(original, nn.Linear):\n            raise TypeError(\"Expected original nn.Linear\")\n        out_dim,in_dim=original.weight.shape\n        rank=int(rank)\n        if rank<=0 or rank>=min(out_dim,in_dim):\n            raise ValueError(f\"Bad SVD rank {rank} for {out_dim}x{in_dim}\")\n        self.in_features=int(in_dim)\n        self.out_features=int(out_dim)\n        self.rank=rank\n        self.A=nn.Parameter(torch.zeros((out_dim,rank),\n            dtype=original.weight.dtype,device=original.weight.device))\n        self.B=nn.Parameter(torch.zeros((rank,in_dim),\n            dtype=original.weight.dtype,device=original.weight.device))\n        if original.bias is None:\n            self.register_parameter(\"bias\",None)\n        else:\n            self.bias=nn.Parameter(original.bias.detach().clone())\n        if A is not None and B is not None:\n            with torch.no_grad():\n                self.A.copy_(A[:,:rank].to(device=self.A.device,dtype=self.A.dtype))\n                self.B.copy_(B[:rank,:].to(device=self.B.device,dtype=self.B.dtype))\n\n    def forward(self,x):\n        return F.linear(F.linear(x,self.B),self.A,self.bias)\n\n\ndef all_encoder_slots(model):\n    \"\"\"Enumerate available projections in EVERY pretrained encoder layer.\"\"\"\n    slots=[]\n    for index,layer in enumerate(model.encoder.layers):\n        for block,names in ((\"attn\",(\"Wqkv\",\"Wo\")),(\"mlp\",(\"Wi\",\"Wo\"))):\n            component=getattr(layer,block)\n            for name in names:\n                module=getattr(component,name,None)\n                if module is None:\n                    raise ValueError(f\"Required projection layer {index}.{block}.{name} missing\")\n                if not isinstance(module,nn.Linear):\n                    raise TypeError(f\"Not an uncompressed nn.Linear: {index}.{block}.{name}: {type(module)}\")\n                slots.append((index,block,name))\n    if len(slots)!=4*len(model.encoder.layers):\n        raise RuntimeError(\"Expected exactly two attention and two FFN projections per layer\")\n    return slots\n\n\ndef list_candidate_families(model):\n    \"\"\"Full brute-force search space: 112 own-SVD + 18 delta-shared slots.\"\"\"\n    options=[{\"layer\":int(layer),\"block\":block,\"component\":name,\n              \"mode\":\"own_svd\"}\n             for layer,block,name in all_encoder_slots(model)]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            options.append({\"layer\":int(second),\"leader\":int(first),\n                            \"block\":\"attn\",\"component\":component,\n                            \"mode\":\"pair_delta\"})\n    return options\n\n\ndef own_svd_factors(original, *, max_rank=256, oversampling=8,niter=2):\n    \"\"\"Randomized, truncated SVD of ORIGINAL pretrained W (not layer delta).\"\"\"\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Expected an uncompressed linear projection\")\n    with torch.no_grad():\n        weight=original.weight.detach().float()\n        rank=min(int(max_rank),min(weight.shape)-1)\n        q=min(rank+int(oversampling),min(weight.shape))\n        U,S,V=torch.svd_lowrank(weight,q=q,niter=int(niter))\n        scale=S[:rank].clamp_min(0).sqrt()\n        A=(U[:,:rank]*scale[None,:]).cpu().contiguous()\n        B=(scale[:,None]*V[:,:rank].T).cpu().contiguous()\n        approximated=A.to(weight.device)@B.to(weight.device)\n        numerator=float((weight-approximated).square().sum().item())\n        denominator=float(weight.square().sum().item())\n    return A,B,{\"rank\":rank,\n                \"relative_frobenius_error\":(numerator/max(denominator,1e-30))**.5}\n\n\ndef projection_at(model, spec):\n    return getattr(getattr(model.encoder.layers[int(spec[\"layer\"])],\n                           str(spec[\"block\"])),str(spec[\"component\"]))\n\n\ndef replace_projection(model, spec, *, factors=None, initialized=True):\n    \"\"\"Apply one projection replacement; return original module for rollback.\"\"\"\n    idx=int(spec[\"layer\"])\n    block=str(spec[\"block\"])\n    comp=str(spec[\"component\"])\n    rank=int(spec[\"rank\"])\n    mode=str(spec[\"mode\"])\n    owner=getattr(model.encoder.layers[idx],block)\n    old=getattr(owner,comp)\n    if not isinstance(old,nn.Linear):\n        raise TypeError(f\"Already compressed or unsupported {idx}.{block}.{comp}\")\n    if mode==\"own_svd\":\n        if initialized and factors is None:\n            factors=own_svd_factors(old,max_rank=rank)[:2]\n        if initialized:\n            A,B=factors\n            replacement=IndependentSVDLinear(old,rank,A,B)\n        else:\n            replacement=IndependentSVDLinear(old,rank)\n    elif mode==\"pair_delta\":\n        if block!=\"attn\" or comp not in (\"Wqkv\",\"Wo\"):\n            raise ValueError(\"Pair delta only supported for local attention\")\n        leader=int(spec[\"leader\"])\n        if (leader,idx) not in local_pair_indices(model):\n            raise ValueError(f\"Not a valid local pair: {leader},{idx}\")\n        shared=getattr(model.encoder.layers[leader].attn,comp)\n        if not isinstance(shared,nn.Linear):\n            raise TypeError(\"Leader must retain original pretrained linear\")\n        if shared.bias is not None or old.bias is not None:\n            raise ValueError(\"Pair delta only supported for bias-free attention\")\n        if initialized:\n            if factors is None:\n                A,B,_=svd_factors(model,leader,idx,comp,max_rank=rank)\n            else:\n                A,B=factors\n            replacement=SVDDeltaLinear(shared,rank,A=A[:,:rank],B=B[:rank,:])\n        else:\n            replacement=SVDDeltaLinear(shared,rank)\n    else:\n        raise ValueError(f\"Unsupported family: {mode}\")\n    setattr(owner,comp,replacement)\n    return old\n\n\ndef revert_projection(model,spec,original):\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Original projection must be nn.Linear\")\n    owner=getattr(model.encoder.layers[int(spec[\"layer\"])],str(spec[\"block\"]))\n    setattr(owner,str(spec[\"component\"]),original)\n\n\ndef applied_adapters(model,selected_specs):\n    result=[]\n    for spec in selected_specs:\n        wrapper=projection_at(model,spec)\n        if not isinstance(wrapper,(IndependentSVDLinear,SVDDeltaLinear)):\n            raise AssertionError(f\"Missing compressed adapter for {spec}\")\n        result.append(wrapper)\n    return result\n\n\ndef expected_saved_bytes(model,spec):\n    \"\"\"FP32 parameter saving for one replacement, including any old bias.\"\"\"\n    idx=int(spec[\"layer\"])\n    owner=getattr(model.encoder.layers[idx],str(spec[\"block\"]))\n    original=getattr(owner,str(spec[\"component\"]))\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Call before installing candidate\")\n    out_dim,in_dim=original.weight.shape\n    rank=int(spec[\"rank\"])\n    # Bias exists in both old and new for own_svd; cancels out.\n    # Pair delta has no bias.\n    return int(4*(out_dim*in_dim-rank*(out_dim+in_dim)))\n\n\ndef load_v50_model(model_dir, *, device=\"cpu\",compile_head=False):\n    \"\"\"Strict reload of all hybrid low-rank projections from local disk only.\"\"\"\n    root=Path(model_dir).resolve()\n    info=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if info[\"format\"]!=\"laya-v50-all28-bruteforce-v1\":\n        raise ValueError(\"Not a V5.0 hybrid model\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(info[\"head_context_budget\"]),\n                         head_layers=int(info[\"head_layers\"]))\n    # Rebuild in the same order: only uncompressed leaders are valid for delta.\n    for spec in info[\"compressed_projections\"]:\n        replace_projection(model,spec,initialized=False)\n    full=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(full,strict=True)\n    for spec,adapter in zip(info[\"compressed_projections\"],\n                            applied_adapters(model,info[\"compressed_projections\"])):\n        if spec[\"mode\"]==\"pair_delta\":\n            leader=getattr(model.encoder.layers[int(spec[\"leader\"])].attn,\n                           str(spec[\"component\"]))\n            assert adapter._shared is leader\n            assert \"_shared\" not in adapter._modules\n    model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tok=AutoTokenizer.from_pretrained(str(root/\"tokenizer\"),local_files_only=True)\n    return model,tok\n\n\n# ================================================================\n# Laya V5.3: CeNN-inspired token-neighborhood liquid recurrence\n# ================================================================\n\nclass LiquidCeNNProjection(nn.Module):\n    \"\"\"Replace/augment an FFN output projection with a 1D liquid CeNN cell.\n\n    This is CeNN-*inspired*: local nearest-neighbor coupling via depthwise\n    Conv1d over tokens plus gated state relaxation. It is not the original\n    continuous-time Chua CeNN equation, nor an LTC ODE solver.\n\n    Modes:\n      * replace: remove the old projection entirely, f(x)=liquid(x).\n      * residual: keep the original projection, f(x)=base(x)+scale*liquid(x).\n\n    The residual scale is initially zero, so this mode is exactly the\n    pretrained baseline at creation time, including identical predictions.\n    \"\"\"\n    def __init__(self,original,width=64,steps=3,mode=\"residual\",\n                 dt_bound=0.50):\n        super().__init__()\n        if not hasattr(original,\"in_features\") or not hasattr(original,\"out_features\"):\n            raise TypeError(\"LiquidCeNN requires a linear-like FFN projection\")\n        self.in_features=int(original.in_features)\n        self.out_features=int(original.out_features)\n        self.width=int(width)\n        self.steps=int(steps)\n        self.mode=str(mode)\n        self.dt_bound=float(dt_bound)\n        if self.width<4 or self.steps<1 or self.steps>10:\n            raise ValueError(\"Bad cell width or recurrent step count\")\n        if self.mode not in (\"replace\",\"residual\"):\n            raise ValueError(self.mode)\n        if not (0.0<self.dt_bound<=1.0):\n            raise ValueError(\"dt_bound must be in (0,1]\")\n        # Only register the original dense/low-rank projection in residual\n        # mode; replacement genuinely deletes its weights from state_dict.\n        self.base=original if self.mode==\"residual\" else None\n        self.drive=nn.Linear(self.in_features,self.width,bias=True)\n        self.drive_norm=nn.LayerNorm(self.width)\n        self.neighbor=nn.Conv1d(self.width,self.width,kernel_size=3,\n                                padding=1,groups=self.width,bias=False)\n        self.recurrent=nn.Linear(self.width,self.width,bias=False)\n        self.liquid_gate=nn.Linear(2*self.width,self.width)\n        self.readout=nn.Linear(self.width,self.out_features,bias=True)\n        # Start with small dynamic corrections instead of random large logits.\n        nn.init.normal_(self.readout.weight,mean=0.,std=0.008)\n        nn.init.zeros_(self.readout.bias)\n        self.scale=nn.Parameter(torch.tensor(0.,dtype=torch.float32))\n\n        # Match original device and dtype without moving the original base.\n        original_parameter=next(original.parameters(),None)\n        if original_parameter is not None:\n            for module in [self.drive,self.drive_norm,self.neighbor,\n                           self.recurrent,self.liquid_gate,self.readout]:\n                module.to(device=original_parameter.device,dtype=original_parameter.dtype)\n\n    def liquid(self,x):\n        # x must have [batch, tokens, d_ffn]. Spatial nearest neighbors\n        # are token neighbors, not a two-dimensional image lattice.\n        if x.ndim!=3:\n            raise ValueError(f\"Expected [B,T,D] FFN activation, got {tuple(x.shape)}\")\n        u=torch.tanh(self.drive_norm(self.drive(x)))\n        h=torch.zeros_like(u)\n        for _ in range(self.steps):\n            neighbors=self.neighbor(h.transpose(1,2)).transpose(1,2)\n            update_gate=torch.sigmoid(self.liquid_gate(torch.cat((u,h),dim=-1)))\n            dt=self.dt_bound*update_gate\n            proposal=torch.tanh(u+neighbors+self.recurrent(h))\n            h=h+dt*(proposal-h)\n        return self.readout(h)\n\n    def forward(self,x):\n        correction=self.liquid(x)\n        if self.mode==\"replace\":\n            return correction\n        # Alpha=0 is an EXACT base-model control despite random CeNN weights.\n        return self.base(x)+self.scale.to(dtype=correction.dtype)*correction\n\n\ndef install_liquid_ffn(model,layer_index,*,width=64,steps=3,mode=\"residual\",\n                       dt_bound=0.5):\n    layer_index=int(layer_index)\n    if not (0<=layer_index<len(model.encoder.layers)):\n        raise IndexError(layer_index)\n    owner=model.encoder.layers[layer_index].mlp\n    previous=owner.Wo\n    if isinstance(previous,LiquidCeNNProjection):\n        raise ValueError(f\"Liquid layer {layer_index} already installed\")\n    candidate=LiquidCeNNProjection(previous,width,steps,mode,dt_bound)\n    owner.Wo=candidate\n    return previous\n\n\ndef uninstall_liquid_ffn(model,layer_index,previous):\n    layer_index=int(layer_index)\n    if not isinstance(previous,nn.Module):\n        raise TypeError(\"Replacement rollback must restore nn.Module\")\n    module=model.encoder.layers[layer_index].mlp\n    if not isinstance(module.Wo,LiquidCeNNProjection):\n        raise ValueError(\"Current layer is not LiquidCeNNProjection\")\n    module.Wo=previous\n\n\ndef find_liquid_cells(model):\n    return [(i,block.mlp.Wo)\n            for i,block in enumerate(model.encoder.layers)\n            if isinstance(block.mlp.Wo,LiquidCeNNProjection)]\n\n\ndef load_v53_model(model_dir, *,device=\"cpu\",compile_head=False):\n    \"\"\"Fully offline strict reconstruction of compressed V5.3 + liquid CeNN.\"\"\"\n    root=Path(model_dir).resolve()\n    metadata=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if metadata.get(\"format\")!=\"laya-v53-liquid-cenn-v1\":\n        raise ValueError(\"Not a V5.3 liquid CeNN export\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(metadata[\"head_context_budget\"]),\n                         head_layers=int(metadata[\"head_layers\"]))\n    for spec in metadata[\"compressed_projections\"]:\n        replace_projection(model,spec,initialized=False)\n    spec=metadata[\"liquid_cell\"]\n    install_liquid_ffn(model,int(spec[\"layer\"]),width=int(spec[\"width\"]),\n       steps=int(spec[\"steps\"]),mode=str(spec[\"mode\"]),\n       dt_bound=float(spec[\"dt_bound\"]))\n    state=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(state,strict=True)\n    assert len(find_liquid_cells(model))==1\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tokenizer=AutoTokenizer.from_pretrained(\n       str(root/\"tokenizer\"),local_files_only=True)\n    return model,tokenizer\n\n\n# ================================================================\n# V5.5 all-28 FFN-output CeNN substitution (no attention substitution)\n# ================================================================\ndef load_v55_model(model_dir, *,device=\"cpu\",compile_head=False):\n    \"\"\"Load fully standalone model with a subset or all 28 FFN liquid cells.\n\n    Each selected encoder layer substitutes its mlp.Wo projection. Attention,\n    residual connections, norms, and all other FFNs remain ModernBERT.\n    \"\"\"\n    root=Path(model_dir).resolve()\n    info=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if info.get(\"format\")!=\"laya-v55-all28-liquid-cenn-v1\":\n        raise ValueError(\"This checkpoint is not an all-layer CeNN V5.5 export\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(info[\"head_context_budget\"]),\n                         head_layers=int(info[\"head_layers\"]))\n    for spec in info[\"compressed_projections\"]:\n        replace_projection(model,spec,initialized=False)\n    liquid_specs=info[\"liquid_cells\"]\n    assert len(liquid_specs)==len({int(s[\"layer\"]) for s in liquid_specs})\n    assert all(s.get(\"mode\")==\"replace\" for s in liquid_specs)\n    for cell in liquid_specs:\n        install_liquid_ffn(model,int(cell[\"layer\"]),\n                          width=int(cell[\"width\"]),\n                          steps=int(cell[\"steps\"]),\n                          mode=\"replace\",dt_bound=float(cell[\"dt_bound\"]))\n    full=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(full,strict=True)\n    assert len(find_liquid_cells(model))==len(liquid_specs)\n    model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tokenizer=AutoTokenizer.from_pretrained(\n        str(root/\"tokenizer\"),local_files_only=True)\n    return model,tokenizer\n"
runtime_file=OUT/"modeling_laya_v55.py"
runtime_file.write_text(V55_RUNTIME_CODE,encoding="utf-8")
spec=importlib.util.spec_from_file_location("v55_runtime",str(runtime_file))
assert spec and spec.loader
v55=importlib.util.module_from_spec(spec)
spec.loader.exec_module(v55)
normalize_question=v55.normalize_question
build_sequence=v55.build_sequence
collate_items=v55.collate_items
QTYPES=v55.QTYPES
teacher,_=v55.load_model(parent_dir,device="cuda")
student,_=v55.load_model(parent_dir,device="cuda")
teacher.eval().requires_grad_(False)
student.eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
student.encoder.config.reference_compile=False
native_unique=v55.unique_parameter_count(teacher)
assert len(student.encoder.layers)==28
assert len(v55.all_encoder_slots(student))==112
# Short liquid cell unit test: residual starts IDENTICAL and true replacement
# physically omits the old base module. Tests B,T,D shapes and gradients.
with torch.enable_grad():
    simple=nn.Linear(80,16,bias=False).to(DEVICE)
    sample=torch.randn(2,11,80,device=DEVICE)
    residual_cell=v55.LiquidCeNNProjection(
        simple,width=8,steps=2,mode="residual").to(DEVICE)
    torch.testing.assert_close(residual_cell(sample),simple(sample),atol=1e-6,rtol=1e-6)
    replacement_cell=v55.LiquidCeNNProjection(
        simple,width=8,steps=2,mode="replace").to(DEVICE)
    assert replacement_cell.base is None
    assert not any(name.startswith("base.") for name in replacement_cell.state_dict())
    result=replacement_cell(sample)
    assert result.shape==(2,11,16) and bool(torch.isfinite(result).all())
    result.square().mean().backward()
    assert replacement_cell.drive.weight.grad is not None
    del simple,sample,residual_cell,replacement_cell,result
gc.collect()
print("PASS: Liquid-CeNN residual identity, replacement weight removal and gradients")


PASS: Liquid-CeNN residual identity, replacement weight removal and gradients


In [4]:
#@title 4. CASE-DISJOINT TRAIN / PROBE / AUDIT benchmark from V4.6
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

# Preserve V5.0's exact case-level TRAIN and PROBE split to reproduce its
# 120/120 step-120 finding. Use the DATASET'S DISTINCT TEST SPLIT for AUDIT.
# Reuse historical V5.1 train/PROBE for comparability, but SELECT using
# unused second-half DEV cases. TEST split stays unseen until after freezing.
rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="train"))
random.Random(SEED).shuffle(rows)
assert TRAIN_CASES+DEV_CASES<=len(rows)
train_rows=rows[:TRAIN_CASES]
historical_probe_rows=rows[TRAIN_CASES:TRAIN_CASES+DEV_CASES//2]
gold_val_rows=rows[TRAIN_CASES+DEV_CASES//2:TRAIN_CASES+DEV_CASES]
train_items=make_items_from_rows(train_rows,SEED,True)
probe_items=make_items_from_rows(historical_probe_rows,SEED+1,True)
gold_val_items=make_items_from_rows(gold_val_rows,SEED+2,True)
# Case namespaces protect against overlapping local split ids.
test_rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="test"))
random.Random(SEED+9001).shuffle(test_rows)
audit_rows=[dict(row,id="TEST::"+str(row.get("id",i)))
            for i,row in enumerate(test_rows)]
audit_items=make_items_from_rows(audit_rows,SEED+9001,True)
assert len(probe_items)>=HISTORICAL_PROBE_DECISIONS
assert len(gold_val_items)>=GOLD_VAL_DECISIONS
assert len(audit_items)>=AUDIT_DECISIONS
groups=[{x["case_id"] for x in seq} for seq in
        (train_items,probe_items,gold_val_items,audit_items)]
for i in range(len(groups)):
    for j in range(i+1,len(groups)):
        assert groups[i].isdisjoint(groups[j]),(i,j)
def direct_batch(selected):
    b=collate_items(selected,int(tokenizer.pad_token_id))
    return {k:v.to(device) for k,v in b.items()}
def batch(items,offset,n=BATCH):
    return direct_batch([items[(offset+i)%len(items)] for i in range(n)])
print("TRAIN:",len(train_items),"OLD PROBE:",len(probe_items),
      "NEW GOLD VAL:",len(gold_val_items),"FINAL TEST:",len(audit_items))


README.md: 0.00B [00:00, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/599k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/222k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]

TRAIN: 2500 OLD PROBE: 600 NEW GOLD VAL: 600 FINAL TEST: 2000


In [5]:
#@title 5. Accuracy, divergence and paired CUDA/wall timing
@torch.inference_mode()
def probs_for(model,b):
    with torch.autocast("cuda",dtype=amp_dtype):
        scores,_=model(**b)
    return torch.softmax(scores.float().masked_fill(
        ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()

@torch.inference_mode()
def get_reference(items,limit):
    out=[]
    for start in range(0,min(limit,len(items)),BATCH):
        chosen=items[start:min(start+BATCH,limit)]
        p=probs_for(teacher,direct_batch(chosen))
        for j,item in enumerate(chosen):
            out.append(dict(probs=p[j],teacher=int(p[j].argmax()),
                gold=int(item["gold_index"]),case=item["case_id"],qid=item["qid"]))
    return out

@torch.inference_mode()
def quality(model,items,ref):
    agree=correct=tcorrect=0;js=0.
    n=len(ref)
    for start in range(0,n,BATCH):
        chosen=items[start:min(n,start+BATCH)]
        p=probs_for(model,direct_batch(chosen))
        for j,item in enumerate(chosen):
            t=ref[start+j]["probs"]; s=p[j]
            assert s.shape==t.shape
            m=(s+t)/2
            js+=float(.5*((s*(s.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()+
                            (t*(t.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()))
            guess=int(s.argmax())
            agree+=int(guess==ref[start+j]["teacher"])
            correct+=int(guess==item["gold_index"])
            tcorrect+=int(ref[start+j]["teacher"]==item["gold_index"])
    return dict(decisions=n,teacher_agreement=agree/n,gold_accuracy=correct/n,
        teacher_gold_accuracy=tcorrect/n,gold_drop=(tcorrect-correct)/n,mean_js=js/n)

BENCH_BATCHES=[batch(probe_items,0,BATCH),batch(probe_items,11,BATCH)]
@torch.inference_mode()
def benchmark_paired(reference,candidate,repeats=BENCH_REPEATS):
    """Alternating A/B/B/A with CUDA events AND synchronized wall-clock timings.
    All timings exclude Colab compilation, dataset loading and model downloading."""
    results=[]
    for j,b in enumerate(BENCH_BATCHES):
        def once(m):
            torch.cuda.synchronize()
            a=torch.cuda.Event(enable_timing=True)
            z=torch.cuda.Event(enable_timing=True)
            before=time.perf_counter()
            a.record()
            with torch.autocast("cuda",dtype=amp_dtype):
                outputs=m(**b)
            z.record();z.synchronize()
            wall=(time.perf_counter()-before)*1000
            return float(a.elapsed_time(z)),float(wall)
        for k in range(4):
            once(reference);once(candidate)
        d={"reference":[],"candidate":[]}
        w={"reference":[],"candidate":[]}
        for k in range(repeats):
            order=("reference","candidate","candidate","reference") if k%2==0 else (
                "candidate","reference","reference","candidate")
            for name in order:
                gpu,cpu=once(reference if name=="reference" else candidate)
                d[name].append(gpu);w[name].append(cpu)
        ref=float(np.median(d["reference"]))
        cand=float(np.median(d["candidate"]))
        rw=float(np.median(w["reference"]))
        cw=float(np.median(w["candidate"]))
        results.append(dict(shape=j,batch=int(b["input_ids"].shape[0]),
            tokens=int(b["input_ids"].shape[1]),
            native_gpu_ms=ref,candidate_gpu_ms=cand,gpu_speedup=ref/max(cand,1e-9),
            native_wall_ms=rw,candidate_wall_ms=cw,wall_speedup=rw/max(cw,1e-9),
            native_p95_ms=float(np.percentile(d["reference"],95)),
            candidate_p95_ms=float(np.percentile(d["candidate"],95)),n=len(d["reference"])))
    return dict(median_gpu_speedup=float(np.median([r["gpu_speedup"] for r in results])),
        min_gpu_speedup=float(min(r["gpu_speedup"] for r in results)),
        median_wall_speedup=float(np.median([r["wall_speedup"] for r in results])),
        min_wall_speedup=float(min(r["wall_speedup"] for r in results)),
        per_shape=results)

# Optional model-quality helpers (aligned to V5.5 configuration).
def valid_quality(q):
    return (q["teacher_agreement"]>=MIN_TEST_AGREEMENT and
            q["gold_drop"]<=MAX_TEST_TEACHER_GOLD_DROP_PP/100 and
            q["mean_js"]<=MAX_JS)
def valid_speed(s):
    return (s["min_gpu_speedup"]>=MIN_SPEED_RATIO and
            s["min_wall_speedup"]>=MIN_SPEED_RATIO)



In [6]:
#@title 6. Reconstruct the exact prior 20-SVD compressed control once
# This exact ordered list comes from the user's V5.0 greedy_combinations.json.
# It replaces the 520-candidate rank sweep; all unchanged projections are
# restored from PINNED pretrained V4.6 weights.
SELECTED_SPECS = [
  {
    "layer": 26,
    "leader": 25,
    "block": "attn",
    "component": "Wqkv",
    "mode": "pair_delta",
    "rank": 64
  },
  {
    "layer": 26,
    "leader": 25,
    "block": "attn",
    "component": "Wo",
    "mode": "pair_delta",
    "rank": 64
  },
  {
    "layer": 17,
    "leader": 16,
    "block": "attn",
    "component": "Wo",
    "mode": "pair_delta",
    "rank": 256
  },
  {
    "layer": 20,
    "leader": 19,
    "block": "attn",
    "component": "Wo",
    "mode": "pair_delta",
    "rank": 256
  },
  {
    "layer": 27,
    "block": "attn",
    "component": "Wqkv",
    "mode": "own_svd",
    "rank": 64
  },
  {
    "layer": 24,
    "block": "mlp",
    "component": "Wi",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 23,
    "block": "attn",
    "component": "Wqkv",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 27,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 32
  },
  {
    "layer": 23,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 32
  },
  {
    "layer": 23,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 25,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 24,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 64
  },
  {
    "layer": 26,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 21,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 21,
    "block": "mlp",
    "component": "Wi",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 20,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 8,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 3,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 2,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 15,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  }
]
assert len(SELECTED_SPECS)==20
assert len({(s["layer"],s["block"],s["component"]) for s in SELECTED_SPECS})==20
assert sum(s["mode"]=="pair_delta" for s in SELECTED_SPECS)==4
# Stable fingerprint of the exact 20 selected projections and ranks.
# Used in model config, provenance and release verification.
spec_hash=hashlib.sha256(
    json.dumps(SELECTED_SPECS,sort_keys=True,separators=(",",":")).encode("utf-8")
).hexdigest()
assert len(spec_hash)==64
print("Compression specification SHA-256:",spec_hash)
def family_key(f):
    return (int(f["layer"]),str(f["block"]),str(f["component"]),str(f["mode"]))
def deterministic_factors(f):
    family_seed=SEED+sum((i+1)*ord(ch) for i,ch in enumerate(
        ":".join(map(str,family_key(f)))))
    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
        torch.manual_seed(family_seed)
        if f["mode"]=="own_svd":
            return v55.own_svd_factors(
                v55.projection_at(student,f),
                max_rank=256,oversampling=8,niter=2)[:2]
        return v55.svd_factors(
            student,int(f["leader"]),int(f["layer"]),
            str(f["component"]),max_rank=256,oversampling=8,niter=2)[:2]

PROBE_REF=get_reference(probe_items,HISTORICAL_PROBE_DECISIONS)
GOLD_VAL_REF=get_reference(gold_val_items,GOLD_VAL_DECISIONS)

native_q=quality(student,probe_items,PROBE_REF)
assert native_q["teacher_agreement"]==1.0,native_q
for j,s in enumerate(SELECTED_SPECS,1):
    factors=deterministic_factors(s)
    old=v55.replace_projection(student,s,factors=factors,initialized=True)
    assert v55.projection_at(student,s) is not old
    print(f"Selected {j:02d}/{len(SELECTED_SPECS)}: layer={s['layer']} "
          f"{s['block']}.{s['component']} {s['mode']} rank={s['rank']}")
unique_student=v55.unique_parameter_count(student)
estimated_bytes=(native_unique-unique_student)*4
assert estimated_bytes>0
initial_probe=quality(student,probe_items,PROBE_REF)
initial_gold_validation=quality(student,gold_val_items,GOLD_VAL_REF)
print("Historical initial PROBE (not used for selection):",initial_probe)
print("New gold-validation baseline:",initial_gold_validation)
print("Teacher gold accuracy in gold-validation:",initial_gold_validation["teacher_gold_accuracy"])

print("Projected parameter savings:",f"{estimated_bytes:,} bytes",
      f"({estimated_bytes/1e6:.3f} MB)")
(OUT/"selected_projections.json").write_text(json.dumps({
    "source_repo":TEACHER_REPO,"source_revision":PINNED_TEACHER_REVISION,
    "specifications":SELECTED_SPECS,"estimated_saved_fp32_bytes":estimated_bytes,
    "initial_probe":initial_probe},indent=2),encoding="utf-8")

# Optional exact, already-gold-tuned V5.2 standalone checkpoint. The config
# must contain the SAME 20 selected projections in the SAME order.
BASELINE_SOURCE="reconstructed_V5.1_frozen_head"
if BASELINE_V52_STANDALONE_DIR.strip():
    root=Path(BASELINE_V52_STANDALONE_DIR.strip()).resolve()
    cfg=json.loads((root/"config.json").read_text(encoding="utf-8"))
    assert cfg["compressed_projections"]==SELECTED_SPECS
    assert cfg["source_revision"]==PINNED_TEACHER_REVISION
    tensors=load_file(str(root/"model.safetensors"),device="cpu")
    student.load_state_dict(tensors,strict=True)
    BASELINE_SOURCE=str(root)
student.eval().requires_grad_(False)
BASELINE_VAL=quality(student,gold_val_items,GOLD_VAL_REF)
BASELINE_OLD_PROBE=quality(student,probe_items,PROBE_REF)
print("CONTROL BASELINE SOURCE:",BASELINE_SOURCE)
print("CONTROL GOLD VALIDATION:",BASELINE_VAL)
(OUT/"baseline_metrics.json").write_text(json.dumps(dict(
    source=BASELINE_SOURCE,teacher_revision=PINNED_TEACHER_REVISION,
    gold_validation=BASELINE_VAL,historical_probe=BASELINE_OLD_PROBE,
    compressed_spec_hash=spec_hash),indent=2))

# IMPORTANT: clone the baseline from the exact compressed state using strict
# state_dict loading (no repeated randomized SVD!). This is used for the final
# paired GPU benchmark and ensures a fair baseline.
baseline_model,_=v55.load_model(parent_dir,device="cuda")
baseline_model.encoder.config.reference_compile=False
baseline_model.eval().requires_grad_(False)
for spec in SELECTED_SPECS:
    v55.replace_projection(baseline_model,spec,initialized=False)
baseline_model.load_state_dict(student.state_dict(),strict=True)
baseline_model.eval().requires_grad_(False)
BASELINE_UNIQUE=v55.unique_parameter_count(baseline_model)
assert BASELINE_UNIQUE==v55.unique_parameter_count(student)
baseline_recheck=quality(baseline_model,gold_val_items,GOLD_VAL_REF)
# The same weights should give the same choices; allow tiny floating-point
# changes in probabilistic divergence on different GPU kernels.
assert baseline_recheck["gold_accuracy"]==BASELINE_VAL["gold_accuracy"]
assert baseline_recheck["teacher_agreement"]==BASELINE_VAL["teacher_agreement"]
assert abs(baseline_recheck["mean_js"]-BASELINE_VAL["mean_js"])<1e-5
print("PASS: compressed baseline exactly reproduced without SVD recomputation")


Compression specification SHA-256: 5cb1804e2e2bea0180c19f3179f3075c9be467e49cc3ac99257f3318f4c6ac6e
Selected 01/20: layer=26 attn.Wqkv pair_delta rank=64
Selected 02/20: layer=26 attn.Wo pair_delta rank=64
Selected 03/20: layer=17 attn.Wo pair_delta rank=256
Selected 04/20: layer=20 attn.Wo pair_delta rank=256
Selected 05/20: layer=27 attn.Wqkv own_svd rank=64
Selected 06/20: layer=24 mlp.Wi own_svd rank=256
Selected 07/20: layer=23 attn.Wqkv own_svd rank=256
Selected 08/20: layer=27 attn.Wo own_svd rank=32
Selected 09/20: layer=23 attn.Wo own_svd rank=32
Selected 10/20: layer=23 mlp.Wo own_svd rank=256
Selected 11/20: layer=25 mlp.Wo own_svd rank=256
Selected 12/20: layer=24 mlp.Wo own_svd rank=64
Selected 13/20: layer=26 mlp.Wo own_svd rank=256
Selected 14/20: layer=21 mlp.Wo own_svd rank=256
Selected 15/20: layer=21 mlp.Wi own_svd rank=256
Selected 16/20: layer=20 mlp.Wo own_svd rank=256
Selected 17/20: layer=8 attn.Wo own_svd rank=256
Selected 18/20: layer=3 attn.Wo own_svd rank=25

In [7]:
#@title 7. Brute-force 28 FFN output layers x all width/step settings
# Standalone pretraining: capture original FFN I/O on TRAIN, NOT gold TEST.
# Candidate quality: one-at-a-time on first 24 gold-VALIDATION decisions.
# Save best CELL STATE per layer, not all 168 redundant neural checkpoints.
assert len(student.encoder.layers)==28
assert len(v55.find_liquid_cells(student))==0
SCREEN_ITEMS=gold_val_items[:SMALL_SCREEN_DECISIONS]
SCREEN_REF=GOLD_VAL_REF[:SMALL_SCREEN_DECISIONS]
BASELINE_SCREEN=quality(baseline_model,SCREEN_ITEMS,SCREEN_REF)
all_trials=[]
best_local={}
selected_states={}
local_layer_models={}
def capture(layer):
    original=student.encoder.layers[layer].mlp.Wo
    recorded=[]
    def hook(_module,args,out):
        recorded.append((args[0].detach().to("cpu",dtype=torch.float16).clone(),
                         out.detach().to("cpu",dtype=torch.float16).clone()))
    handle=original.register_forward_hook(hook)
    try:
        for j in range(CAPTURE_BATCHES):
            b=batch(train_items,(j*BATCH+layer*7)%len(train_items),BATCH)
            with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
                student(**b)
    finally:
        handle.remove()
    assert len(recorded)==CAPTURE_BATCHES
    return original,recorded

def local_score(entry):
    q=entry["single_layer_screen"]
    # Gold correctness first; KL/JS second; saving third.
    return (q["gold_accuracy"],q["teacher_agreement"],
            -q["mean_js"],entry["parameter_saving"],
            -entry["last_relative_mse"])

for layer in range(28):
    original,examples=capture(layer)
    original_params=sum(p.numel() for p in original.parameters())
    trials_for_layer=[]
    for width in SEARCH_WIDTHS:
        for steps in SEARCH_STEPS:
            candidate=v55.LiquidCeNNProjection(original,width=width,steps=steps,
                mode="replace",dt_bound=CE_DT_BOUND).to(DEVICE,dtype=torch.float32)
            new_params=sum(p.numel() for p in candidate.parameters())
            if new_params>=original_params:
                all_trials.append(dict(layer=layer,width=width,steps=steps,
                     parameter_saving=original_params-new_params,
                     skipped="nonpositive_saving"))
                del candidate
                continue
            opt=torch.optim.AdamW(candidate.parameters(),
                                  lr=LOCAL_DISTILL_LR,weight_decay=0.)
            losses=[]
            for iteration in range(LOCAL_DISTILL_STEPS):
                a,z=examples[iteration%len(examples)]
                x=a.to(DEVICE,dtype=torch.float32)
                target=z.to(DEVICE,dtype=torch.float32)
                candidate.train()
                opt.zero_grad(set_to_none=True)
                pred=candidate(x)
                lmse=F.mse_loss(pred,target)/target.square().mean().clamp_min(1e-7)
                assert bool(torch.isfinite(lmse))
                lmse.backward()
                torch.nn.utils.clip_grad_norm_(candidate.parameters(),1.0)
                opt.step()
                if iteration==0 or (iteration+1)%10==0:
                    losses.append(float(lmse.detach().cpu()))
            candidate.eval().requires_grad_(False)
            owner=student.encoder.layers[layer].mlp
            owner.Wo=candidate
            try:
                one=quality(student,SCREEN_ITEMS,SCREEN_REF)
            finally:
                owner.Wo=original
            result=dict(layer=layer,width=int(width),steps=int(steps),
                original_parameters=original_params,new_parameters=new_params,
                parameter_saving=original_params-new_params,
                last_relative_mse=losses[-1],single_layer_screen=one,
                relative_mse_trace=losses)
            all_trials.append(result)
            trials_for_layer.append(result)
            key=(layer,width,steps)
            local_layer_models[key]={k:v.detach().cpu().contiguous().clone()
                                     for k,v in candidate.state_dict().items()}
            del candidate,opt
    assert trials_for_layer,f"No lower-parameter liquid candidate for FFN layer {layer}"
    best=max(trials_for_layer,key=local_score)
    best_local[layer]=best
    selected_states[layer]=local_layer_models[
       (layer,best["width"],best["steps"])]
    for k in list(local_layer_models):
        if k[0]==layer:
            del local_layer_models[k]
    print(f"LAYER {layer:02d}/27: best width={best['width']}, steps={best['steps']} "
          f"screen_gold={best['single_layer_screen']['gold_accuracy']:.3f}, "
          f"JS={best['single_layer_screen']['mean_js']:.5f}, "
          f"FP32 saving={best['parameter_saving']*4/1e6:.3f}MB")
    (OUT/"all28_bruteforce_trials.json").write_text(
        json.dumps(dict(model=TEACHER_REPO,revision=PINNED_TEACHER_REVISION,
            candidate_widths=SEARCH_WIDTHS,candidate_steps=SEARCH_STEPS,
            completed_layers=layer+1,trials=all_trials,
            best_per_layer=list(best_local.values())),indent=2))
    del original,examples
    gc.collect()
assert len(best_local)==28 and len(selected_states)==28
print("Brute force DONE:",len(all_trials),"settings evaluated or skipped")


LAYER 00/27: best width=64, steps=1 screen_gold=0.458, JS=0.11033, FP32 saving=9.759MB
LAYER 01/27: best width=64, steps=2 screen_gold=0.958, JS=0.00642, FP32 saving=9.759MB
LAYER 02/27: best width=64, steps=1 screen_gold=0.833, JS=0.00622, FP32 saving=9.759MB
LAYER 03/27: best width=32, steps=2 screen_gold=0.833, JS=0.00458, FP32 saving=10.264MB
LAYER 04/27: best width=64, steps=2 screen_gold=0.917, JS=0.00739, FP32 saving=9.759MB
LAYER 05/27: best width=16, steps=1 screen_gold=0.833, JS=0.01207, FP32 saving=10.507MB
LAYER 06/27: best width=32, steps=2 screen_gold=0.833, JS=0.00580, FP32 saving=10.264MB
LAYER 07/27: best width=64, steps=2 screen_gold=0.917, JS=0.00521, FP32 saving=9.759MB
LAYER 08/27: best width=16, steps=2 screen_gold=0.917, JS=0.00469, FP32 saving=10.507MB
LAYER 09/27: best width=64, steps=2 screen_gold=0.875, JS=0.00732, FP32 saving=9.759MB
LAYER 10/27: best width=32, steps=2 screen_gold=0.917, JS=0.00907, FP32 saving=10.264MB
LAYER 11/27: best width=64, steps=2 sc

In [8]:
#@title 8. FORCE INSTALL all 28 liquid FFNs + cumulative quality trajectory
# This is a genuine ALL-28 FFN projection substitution. Attention and
# other model blocks stay the same. Cumulative results use VALIDATION only.
old_projections={}
cumulative=[]
for layer in range(28):
    spec=best_local[layer]
    old=v55.install_liquid_ffn(student,layer,
        width=int(spec["width"]),steps=int(spec["steps"]),
        mode="replace",dt_bound=CE_DT_BOUND)
    old_projections[layer]=old
    cell=student.encoder.layers[layer].mlp.Wo
    cell.load_state_dict(selected_states[layer],strict=True)
    q=quality(student,gold_val_items,GOLD_VAL_REF)
    cumulative.append(dict(replaced_count=layer+1,
        added_layer=layer,gold_accuracy=q["gold_accuracy"],
        teacher_agreement=q["teacher_agreement"],mean_js=q["mean_js"],
        teacher_gold_accuracy=q["teacher_gold_accuracy"],
        width=spec["width"],steps=spec["steps"]))
    print(f"FORCED {layer+1:02d}/28: gold={q['gold_accuracy']:.3%}, "
          f"teacher_agreement={q['teacher_agreement']:.3%}, JS={q['mean_js']:.5f}")
    (OUT/"all28_cumulative_quality.json").write_text(json.dumps(
        dict(original_baseline=BASELINE_VAL,
             forced_28_cumulative=cumulative),indent=2))
assert len(v55.find_liquid_cells(student))==28
ALL28_VAL_INITIAL=quality(student,gold_val_items,GOLD_VAL_REF)
print("ALL28 INITIAL VALIDATION:",ALL28_VAL_INITIAL)


FORCED 01/28: gold=49.444%, teacher_agreement=55.000%, JS=0.08521
FORCED 02/28: gold=42.222%, teacher_agreement=43.889%, JS=0.09676
FORCED 03/28: gold=38.333%, teacher_agreement=36.667%, JS=0.10825
FORCED 04/28: gold=27.222%, teacher_agreement=29.444%, JS=0.10794
FORCED 05/28: gold=31.111%, teacher_agreement=30.556%, JS=0.10832
FORCED 06/28: gold=35.556%, teacher_agreement=32.222%, JS=0.10843
FORCED 07/28: gold=31.667%, teacher_agreement=33.333%, JS=0.11011
FORCED 08/28: gold=33.889%, teacher_agreement=33.333%, JS=0.10542
FORCED 09/28: gold=30.000%, teacher_agreement=31.111%, JS=0.10848
FORCED 10/28: gold=20.556%, teacher_agreement=21.111%, JS=0.10816
FORCED 11/28: gold=26.111%, teacher_agreement=28.333%, JS=0.10817
FORCED 12/28: gold=32.778%, teacher_agreement=35.556%, JS=0.10781
FORCED 13/28: gold=27.222%, teacher_agreement=27.222%, JS=0.10812
FORCED 14/28: gold=27.222%, teacher_agreement=27.778%, JS=0.10783
FORCED 15/28: gold=31.111%, teacher_agreement=31.111%, JS=0.10765
FORCED 16/

In [9]:
#@title 9. Gold + KL joint train ALL28, retain best validation checkpoint
# All 28 LiquidCeNN layers must stay installed during these updates.
assert len(v55.find_liquid_cells(student))==28
student.requires_grad_(False)
for _,cell in v55.find_liquid_cells(student):
    for p in cell.parameters():
        p.requires_grad_(True)
trainable=[p for p in student.parameters() if p.requires_grad]
assert trainable
trainable_names={k for k,p in student.named_parameters() if p.requires_grad}
student.eval()  # Disable dropout, keep backprop enabled.
BEST_WEIGHTS=OUT/"best_all28_trainable.safetensors"
TRAINING_LOG=OUT/"all28_training_progress.json"
def save_all28(step,q,reason):
    from collections import OrderedDict
    tensor_map=OrderedDict((k,p.detach().cpu().contiguous().clone())
        for k,p in student.named_parameters() if p.requires_grad)
    save_file(tensor_map,str(BEST_WEIGHTS),metadata={
        "format":"pt","selected_step":str(step),"reason":reason})
    (OUT/"best_all28_trainable.json").write_text(json.dumps(
        dict(step=step,quality=q,reason=reason,
            active_liquid_cell_count=len(v55.find_liquid_cells(student))),indent=2))
def restore_all28():
    weights=load_file(str(BEST_WEIGHTS),device="cpu")
    assert set(weights)==trainable_names
    with torch.no_grad():
        for k,p in student.named_parameters():
            if k in weights:
                p.copy_(weights[k].to(p.device,dtype=p.dtype))
def score_quality(q):
    # Gold first, agreement second, JS third; don't prefer a worse gold
    # checkpoint just because KL happens to be smaller.
    return (q["gold_accuracy"],q["teacher_agreement"],-q["mean_js"])
train_rng=random.Random(SEED+550)
train_order=list(range(len(train_items)))
train_rng.shuffle(train_order)
train_cursor=0
def get_training_batch():
    global train_cursor
    indexes=[]
    for _ in range(TRAIN_B):
        if train_cursor>=len(train_order):
            train_rng.shuffle(train_order)
            train_cursor=0
        indexes.append(train_order[train_cursor])
        train_cursor+=1
    selected=[train_items[j] for j in indexes]
    b=direct_batch(selected)
    assert bool((b["marker_pos"][b["marker_mask"]]<TRAIN_MAX_TOKENS).all())
    for key in ("input_ids","attention_mask"):
        b[key]=b[key][:,:TRAIN_MAX_TOKENS]
    gold=torch.tensor([x["gold_index"] for x in selected],
            dtype=torch.long,device=DEVICE)
    soft=torch.zeros_like(b["marker_mask"],dtype=torch.float32,device=DEVICE)
    for i,x in enumerate(selected):
        pp=torch.tensor(x["gold_probs"],dtype=torch.float32,device=DEVICE)
        soft[i,:len(pp)]=pp/pp.sum().clamp_min(1e-8)
    return b,gold,soft
best_quality=ALL28_VAL_INITIAL
best_step=0
history=[]
save_all28(0,best_quality,"forced_28_initial")
optimizer=torch.optim.AdamW(trainable,lr=TRAIN_LR,weight_decay=0.)
scaler=torch.amp.GradScaler("cuda",enabled=amp_dtype==torch.float16)
patience=0
for step in range(1,FORCE_ALL28_TRAIN_STEPS+1):
    inputs,gold,soft=get_training_batch()
    with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
        teacher_logits,_=teacher(**inputs)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda",dtype=amp_dtype):
        candidate_logits,_=student(**inputs)
    valid=inputs["marker_mask"].bool()
    scores=candidate_logits.float().masked_fill(~valid,-1e9)
    reference=teacher_logits.float().masked_fill(~valid,-1e9)
    logs=F.log_softmax(scores,dim=-1)
    loss_ce=F.nll_loss(logs,gold)
    loss_soft=-(soft*logs).sum(-1).mean()
    loss_teacher=F.kl_div(F.log_softmax(scores/2,dim=-1),
        F.softmax(reference/2,dim=-1),reduction="batchmean")*4.
    loss=(TRAIN_GOLD_CE*loss_ce+
          TRAIN_SOFT_GOLD*loss_soft+TRAIN_TEACHER_KL*loss_teacher)
    if not bool(torch.isfinite(loss)):
        raise RuntimeError(f"All28 training produced a non-finite loss at {step}")
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(trainable,1.0)
    scaler.step(optimizer);scaler.update()
    if step%TRAIN_CHECK_EVERY==0 or step==FORCE_ALL28_TRAIN_STEPS:
        q=quality(student,gold_val_items,GOLD_VAL_REF)
        improved=score_quality(q)>score_quality(best_quality)
        if improved:
            best_quality=q;best_step=step;patience=0
            save_all28(step,q,"best_forced_all28_gold")
        else:
            patience+=1
        history.append(dict(step=step,loss=float(loss.detach().cpu()),
            gold_ce=float(loss_ce.detach().cpu()),
            gold_soft=float(loss_soft.detach().cpu()),
            teacher_kl=float(loss_teacher.detach().cpu()),
            quality=q,improved=improved))
        TRAINING_LOG.write_text(json.dumps(history,indent=2))
        print(f"ALL28 TRAIN step={step} gold={q['gold_accuracy']:.3%}, "
              f"agreement={q['teacher_agreement']:.3%} selected={improved}")
        if patience>=4:
            print("EARLY STOP: four checks without validation improvement")
            break
restore_all28()
student.eval().requires_grad_(False)
ALL28_VAL_FINAL=quality(student,gold_val_items,GOLD_VAL_REF)
assert ALL28_VAL_FINAL["gold_accuracy"]==best_quality["gold_accuracy"]
assert len(v55.find_liquid_cells(student))==28
(OUT/"all28_training_summary.json").write_text(json.dumps(dict(
    all28_initial=ALL28_VAL_INITIAL,all28_after_training=ALL28_VAL_FINAL,
    baseline_validation=BASELINE_VAL,
    selected_step=best_step,training_progress=history,
    total_liquid_cells=28),indent=2))


ALL28 TRAIN step=20 gold=27.222%, agreement=26.111% selected=False
ALL28 TRAIN step=40 gold=35.556%, agreement=35.000% selected=True
ALL28 TRAIN step=60 gold=38.889%, agreement=40.000% selected=True
ALL28 TRAIN step=80 gold=35.000%, agreement=31.667% selected=False
ALL28 TRAIN step=100 gold=35.000%, agreement=33.889% selected=False
ALL28 TRAIN step=120 gold=35.000%, agreement=32.778% selected=False
ALL28 TRAIN step=140 gold=32.778%, agreement=31.667% selected=False
EARLY STOP: four checks without validation improvement


3999

In [10]:
#@title 10. Greedy quality-gated SUBSET ablation vs forced all28 — NO TEST selection
# Start from compressed baseline in the same student after preserving forced
# all-28 weights (no extra 1.5GB model clone). Greedy accepts only candidates
# that preserve 180-validation gold accuracy and teacher agreement.
full28_weights={i:{k:v.detach().cpu().clone()
  for k,v in student.encoder.layers[i].mlp.Wo.state_dict().items()}
  for i in range(28)}
FORCED28_SPECS=[dict(layer=i,width=int(best_local[i]["width"]),
 steps=int(best_local[i]["steps"]),mode="replace",dt_bound=CE_DT_BOUND)
 for i in range(28)]
ALL28_PARAMS=v55.unique_parameter_count(student)
for i in range(28):
    v55.uninstall_liquid_ffn(student,i,old_projections[i])
assert len(v55.find_liquid_cells(student))==0
reset_q=quality(student,gold_val_items,GOLD_VAL_REF)
assert reset_q["gold_accuracy"]==BASELINE_VAL["gold_accuracy"]
assert reset_q["teacher_agreement"]==BASELINE_VAL["teacher_agreement"]
del old_projections
gc.collect()
selected_subset=[]
subset_trajectory=[]
for item in sorted(best_local.values(),key=lambda r:(
    -r["single_layer_screen"]["gold_accuracy"],
    r["single_layer_screen"]["mean_js"],
    -r["parameter_saving"])):
    layer=int(item["layer"])
    prior=v55.install_liquid_ffn(student,layer,width=int(item["width"]),
        steps=int(item["steps"]),mode="replace",dt_bound=CE_DT_BOUND)
    layer_cell=student.encoder.layers[layer].mlp.Wo
    layer_cell.load_state_dict(selected_states[layer],strict=True)
    q=quality(student,gold_val_items,GOLD_VAL_REF)
    gold_ok=100*(BASELINE_VAL["gold_accuracy"]-q["gold_accuracy"])<=(
        GREEDY_MAX_VAL_GOLD_DROP_PP+1e-8)
    agreement_ok=q["teacher_agreement"]>=GREEDY_MIN_AGREEMENT
    accepted=bool(gold_ok and agreement_ok and q["mean_js"]<=MAX_JS)
    if accepted:
        selected_subset.append(dict(layer=layer,width=int(item["width"]),
            steps=int(item["steps"]),mode="replace",dt_bound=CE_DT_BOUND))
    else:
        v55.uninstall_liquid_ffn(student,layer,prior)
    subset_trajectory.append(dict(layer=layer,accepted=accepted,
        trial_validation=q,accepted_total=len(selected_subset)))
    print(f"GREEDY layer={layer:02d}, accepted={accepted}, "
          f"total={len(selected_subset)}, gold={q['gold_accuracy']:.3%}")
    (OUT/"greedy_accepted_subset.json").write_text(json.dumps(dict(
        baseline=BASELINE_VAL,selected=selected_subset,
        history=subset_trajectory),indent=2))
SUBSET_VAL=quality(student,gold_val_items,GOLD_VAL_REF)
assert len(v55.find_liquid_cells(student))==len(selected_subset)
print("SUBSET VALIDATION:",SUBSET_VAL)
# NOTE: The subset model is evaluated separately; it is NEVER renamed ALL28.


GREEDY layer=20, accepted=True, total=1, gold=85.556%
GREEDY layer=16, accepted=False, total=1, gold=82.778%
GREEDY layer=01, accepted=False, total=1, gold=82.778%
GREEDY layer=19, accepted=False, total=1, gold=82.778%
GREEDY layer=24, accepted=True, total=2, gold=85.000%
GREEDY layer=23, accepted=True, total=3, gold=85.000%
GREEDY layer=26, accepted=True, total=4, gold=84.444%
GREEDY layer=08, accepted=True, total=5, gold=84.444%
GREEDY layer=07, accepted=False, total=5, gold=81.667%
GREEDY layer=13, accepted=False, total=5, gold=77.778%
GREEDY layer=04, accepted=False, total=5, gold=81.667%
GREEDY layer=10, accepted=False, total=5, gold=75.556%
GREEDY layer=27, accepted=False, total=5, gold=84.444%
GREEDY layer=21, accepted=False, total=5, gold=85.000%
GREEDY layer=22, accepted=False, total=5, gold=85.556%
GREEDY layer=25, accepted=False, total=5, gold=84.444%
GREEDY layer=18, accepted=False, total=5, gold=84.444%
GREEDY layer=15, accepted=False, total=5, gold=83.333%
GREEDY layer=09

In [11]:
#@title 11. One-time TEST results for forced 28, greedy subset and baseline; GPU speed
# To avoid a 4th full model copy, compare subset first, restore all 28 exact
# trained weights, then evaluate forced 28. No model choice uses TEST metrics.
TEST_REF=get_reference(audit_items,AUDIT_DECISIONS)
assert {r["case"] for r in TEST_REF}.isdisjoint(
    {r["case"] for r in GOLD_VAL_REF})
base_test=quality(baseline_model,audit_items,TEST_REF)
subset_test=quality(student,audit_items,TEST_REF)
BENCH_OLD=BENCH_BATCHES
BENCH_BATCHES=[batch(audit_items,0,BATCH),batch(audit_items,11,BATCH)]
subset_speed=benchmark_paired(baseline_model,student,
    repeats=max(BENCH_REPEATS,15))
BENCH_BATCHES=BENCH_OLD
# Recover EXACT 28-FFN model trained earlier: restore original compressed
# modules from cloned baseline without changing original SVD factors.
for spec in selected_subset:
    idx=spec["layer"]
    student.encoder.layers[idx].mlp.Wo=baseline_model.encoder.layers[idx].mlp.Wo
# IMPORTANT: this assignment copies nn.Module objects by reference but they
# are part of a frozen baseline and will be replaced immediately below.
assert len(v55.find_liquid_cells(student))==0
for spec in FORCED28_SPECS:
    layer=spec["layer"]
    v55.install_liquid_ffn(student,layer,
        width=spec["width"],steps=spec["steps"],mode="replace",
        dt_bound=spec["dt_bound"])
    student.encoder.layers[layer].mlp.Wo.load_state_dict(
        full28_weights[layer],strict=True)
assert len(v55.find_liquid_cells(student))==28
student.eval().requires_grad_(False)
forced_recheck=quality(student,gold_val_items,GOLD_VAL_REF)
assert forced_recheck["gold_accuracy"]==ALL28_VAL_FINAL["gold_accuracy"]
forced_test=quality(student,audit_items,TEST_REF)
BENCH_BATCHES=[batch(audit_items,0,BATCH),batch(audit_items,11,BATCH)]
forced_speed=benchmark_paired(baseline_model,student,
    repeats=max(BENCH_REPEATS,15))
BENCH_BATCHES=BENCH_OLD
test_comparison=dict(
 original_teacher_accuracy=forced_test["teacher_gold_accuracy"],
 compressed_baseline_accuracy=base_test["gold_accuracy"],
 forced_28_liquid_accuracy=forced_test["gold_accuracy"],
 accepted_subset_accuracy=subset_test["gold_accuracy"],
 forced_28_metrics=forced_test,subset_metrics=subset_test,
 baseline_metrics=base_test,forced_28_gpu_and_wall_speed=forced_speed,
 subset_gpu_and_wall_speed=subset_speed,
 selected_subset_count=len(selected_subset),
 forced_28_count=len(v55.find_liquid_cells(student)))
(OUT/"all28_test_comparison.json").write_text(
    json.dumps(test_comparison,indent=2))
print("TEST REPORT:\n",json.dumps(dict(
  teacher=forced_test["teacher_gold_accuracy"],
  baseline=base_test["gold_accuracy"],
  forced_all28=forced_test["gold_accuracy"],
  subset=subset_test["gold_accuracy"],
  greedy_count=len(selected_subset),
  forced_speed=forced_speed["median_gpu_speedup"],
  subset_speed=subset_speed["median_gpu_speedup"]),indent=2))


TEST REPORT:
 {
  "teacher": 0.765,
  "baseline": 0.755,
  "forced_all28": 0.31,
  "subset": 0.77,
  "greedy_count": 5,
  "forced_speed": 0.8447844630509354,
  "subset_speed": 0.9741199839373174
}


In [12]:
#@title 12. Offline full ALL28 model export, strict reload, publication quality gates
from collections import OrderedDict
assert len(v55.find_liquid_cells(student))==28
(DEST/"encoder").mkdir(parents=True,exist_ok=True)
(DEST/"tokenizer").mkdir(parents=True,exist_ok=True)
student.encoder.config.save_pretrained(str(DEST/"encoder"))
tokenizer.save_pretrained(str(DEST/"tokenizer"))
(DEST/"modeling_laya_v55.py").write_text(V55_RUNTIME_CODE,encoding="utf-8")
config=dict(format="laya-v55-all28-liquid-cenn-v1",
  model_type="laya-v55-all28-liquid-cenn",
  source_repo=TEACHER_REPO,source_revision=PINNED_TEACHER_REVISION,
  compressed_projections=SELECTED_SPECS,
  head_context_budget=64,head_layers=int(source_config["head_layers"]),
  liquid_cells=FORCED28_SPECS,liquid_projection_scope="FFN mlp.Wo ONLY",
  converted_ffn_projections=28,attention_layers_converted=0,
  selected_training_step=best_step)
(DEST/"config.json").write_text(json.dumps(config,indent=2))
weights=DEST/"model.safetensors"
save_file(OrderedDict((k,v.detach().cpu().contiguous().clone())
     for k,v in student.state_dict().items()),str(weights),
     metadata={"format":"pt","architecture":"laya-v55-all28-liquid-cenn"})
orig_bytes=(parent_dir/"model.safetensors").stat().st_size
model_bytes=weights.stat().st_size
actual_saved_bytes=orig_bytes-model_bytes
reloaded,rtok=v55.load_v55_model(DEST,device="cuda")
example=batch(gold_val_items,0,BATCH)
with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
    output,_=student(**example)
    loaded,_=reloaded(**example)
max_abs_reload_diff=float((output.float()-loaded.float()).abs().max())
reload_labels_identical=bool(torch.equal(output.argmax(-1),loaded.argmax(-1)))
reload_close=bool(torch.allclose(output.float(),loaded.float(),
     atol=2e-3,rtol=2e-3))
assert reload_labels_identical and reload_close
assert len(v55.find_liquid_cells(reloaded))==28
del reloaded
gc.collect();torch.cuda.empty_cache()
test_loss_vs_teacher_pp=100*(
    forced_test["teacher_gold_accuracy"]-forced_test["gold_accuracy"])
test_regression_vs_baseline_pp=100*(
    base_test["gold_accuracy"]-forced_test["gold_accuracy"])
speed_gate=(forced_speed["min_gpu_speedup"]>=MIN_SPEED_RATIO
    and forced_speed["min_wall_speedup"]>=MIN_SPEED_RATIO)
release_gates=dict(
 all28_replacements_installed=len(FORCED28_SPECS)==28,
 full_validation_not_below_baseline=
    ALL28_VAL_FINAL["gold_accuracy"]>=BASELINE_VAL["gold_accuracy"]-1e-8,
 test_not_below_compressed_baseline=
    test_regression_vs_baseline_pp<=1e-8,
 gold_test_loss_vs_teacher_within_limit=
    test_loss_vs_teacher_pp<=MAX_TEST_TEACHER_GOLD_DROP_PP,
 test_teacher_agreement=forced_test["teacher_agreement"]>=MIN_TEST_AGREEMENT,
 test_JS=forced_test["mean_js"]<=MAX_JS,
 storage_at_least_100MB_below_teacher=
    actual_saved_bytes>=MIN_TOTAL_FP32_SAVED_BYTES,
 speed_nonregression_bounded=speed_gate,
 strict_offline_reload=reload_labels_identical and reload_close)
release_gates["all_pass"]=all(release_gates.values())
report=dict(version="V5.5",scope="all 28 FFN Wo outputs replaced, not attention",
  full28_specs=FORCED28_SPECS,selected_subset=selected_subset,
  selection_trails=subset_trajectory,all_trials=all_trials,
  cumulative_validation=cumulative,all28_final_validation=ALL28_VAL_FINAL,
  all28_test=forced_test,subset_test=subset_test,
  compressed_baseline_test=base_test,
  comparison=test_comparison,
  teacher_FP32_weight_bytes=orig_bytes,
  all28_FP32_weight_bytes=model_bytes,
  storage_savings_from_teacher_bytes=actual_saved_bytes,
  last_model_reload_max_abs_diff=max_abs_reload_diff,
  release_gates=release_gates,
  limitation="The 200-decision TEST dataset has appeared in previous experiments; this is not an untouched cross-study holdout.")
(DEST/"v55_report.json").write_text(json.dumps(report,indent=2))
readme=f"""---
library_name: pytorch
base_model: {TEACHER_REPO}
tags:
- modernbert
- liquid-cenn
- all28
- experimental
---

# Laya V5.5 — All 28 FFN projections replaced by Liquid CeNN

**28 of 28 FFN output projections (mlp.Wo) were replaced.** ModernBERT
attention, normalization and the original scoring head are still present.
Each layer has a CeNN-inspired liquid token-neighborhood recurrent module.
This is NOT a full attention-to-CeNN transformation.

## Observed evaluation (200 typed TEST decisions)

- Original teacher gold accuracy: **{forced_test['teacher_gold_accuracy']:.3%}**
- Compressed baseline gold accuracy: **{base_test['gold_accuracy']:.3%}**
- Fully replaced 28-liquid model accuracy: **{forced_test['gold_accuracy']:.3%}**
- Greedy partial model gold accuracy: **{subset_test['gold_accuracy']:.3%}** ({len(selected_subset)} liquid substitutions)
- Forced full-28 gold validation: **{ALL28_VAL_FINAL['gold_accuracy']:.3%}**
- Full-28 teacher agreement on TEST: **{forced_test['teacher_agreement']:.3%}**
- GPU speed ratio vs compressed baseline: **{forced_speed['median_gpu_speedup']:.3f}x**
- FP32 checkpoint: **{model_bytes:,} bytes**
- FP32 savings vs original teacher: **{actual_saved_bytes:,} bytes**
- Release gate result: **{release_gates['all_pass']}**

Results are limited to this small benchmark and do not imply that
standard attention was replaced or that generalized performance improved.

## Offline model load

\`\`\`python
from modeling_laya_v55 import load_v55_model
model,tokenizer=load_v55_model("/path/to/local/checkpoint",device="cuda")
\`\`\`
"""
(DEST/"README.md").write_text(readme,encoding="utf-8")
(DEST/"requirements.txt").write_text(
    "torch>=2.4\ntransformers==4.57.6\nsafetensors>=0.4\nhuggingface_hub>=0.25,<1\n")
(DEST/"provenance.json").write_text(json.dumps(dict(
 source=TEACHER_REPO,revision=PINNED_TEACHER_REVISION,
 liquid_cells=FORCED28_SPECS,compressed_projections=SELECTED_SPECS,
 never_claims_attention_CeNN=True),indent=2))
HF_STATUS="blocked_by_quality_gates"
HF_ERROR=None
HF_URL=None
if release_gates["all_pass"] and PUBLISH_TO_HF:
    try:
        publisher=HfApi()
        try:
            who=publisher.whoami()["name"]
        except Exception:
            notebook_login()
            who=publisher.whoami()["name"]
        assert who=="vtava",f"Expected Hugging Face vtava account, got {who}"
        assert STUDENT_REPO=="vtava/Laya-v55-all28-liquid-cenn"
        if publisher.repo_exists(STUDENT_REPO,repo_type="model"):
            remote=set(publisher.list_repo_files(STUDENT_REPO,repo_type="model"))
            if remote-{".gitattributes",".gitignore"} and not ALLOW_UPDATE_EXISTING:
                raise RuntimeError("Existing model populated; no implicit overwrite")
        else:
            publisher.create_repo(STUDENT_REPO,repo_type="model",
                private=PRIVATE_HF_REPO,exist_ok=False)
        publisher.upload_large_folder(repo_id=STUDENT_REPO,
            repo_type="model",folder_path=str(DEST))
        complete=set(publisher.list_repo_files(STUDENT_REPO,repo_type="model"))
        expected={"model.safetensors","config.json","modeling_laya_v55.py",
            "encoder/config.json","v55_report.json","README.md"}
        assert expected.issubset(complete),f"HF incomplete: {expected-complete}"
        HF_STATUS="published"
        HF_URL=f"https://huggingface.co/{STUDENT_REPO}"
        print("FULL 28 MODEL PUBLISHED:",HF_URL)
    except Exception as err:
        HF_STATUS="upload_error"
        HF_ERROR=f"{type(err).__name__}: {err}"
        import traceback
        (OUT/"hf_upload_error.txt").write_text(traceback.format_exc())
elif release_gates["all_pass"]:
    HF_STATUS="disabled_by_user"
(OUT/"huggingface_status.json").write_text(json.dumps(dict(
    status=HF_STATUS,error=HF_ERROR,url=HF_URL,
    all28=28,all_gates_pass=release_gates["all_pass"],
    release_gates=release_gates),indent=2))
print("HF STATUS:",HF_STATUS)
print("RELEASE GATES:",json.dumps(release_gates,indent=2))


<>:119: SyntaxWarning: invalid escape sequence '\`'
<>:119: SyntaxWarning: invalid escape sequence '\`'
/tmp/ipykernel_2077/837437994.py:119: SyntaxWarning: invalid escape sequence '\`'
  never_claims_attention_CeNN=True),indent=2))


HF STATUS: blocked_by_quality_gates
RELEASE GATES: {
  "all28_replacements_installed": true,
  "full_validation_not_below_baseline": false,
  "test_not_below_compressed_baseline": false,
  "gold_test_loss_vs_teacher_within_limit": false,
  "test_teacher_agreement": false,
  "test_JS": false,
  "storage_at_least_100MB_below_teacher": true,
  "speed_nonregression_bounded": false,
  "strict_offline_reload": true,
  "all_pass": false
}


In [13]:
#@title 13. Download STATISTICS ONLY — full model never downloaded to browser
import json,zipfile,csv,io,hashlib
from pathlib import Path
from datetime import datetime,timezone
if "OUT" not in globals():
    OUT=Path("/content/Laya_V55_All28_LiquidCeNN")
OUT=Path(OUT).resolve()
ZIP=OUT.parent/"Laya_V55_All28_Statistics.zip"
summ=dict(
 version="V5.5",date_utc=datetime.now(timezone.utc).isoformat(),
 scope="All28 CeNN FFN Wo, not attention",
 forced_all28_gold_validation=globals().get("ALL28_VAL_FINAL",{}).get("gold_accuracy"),
 forced_all28_gold_test=globals().get("forced_test",{}).get("gold_accuracy"),
 baseline_gold_test=globals().get("base_test",{}).get("gold_accuracy"),
 selected_partial_count=len(globals().get("selected_subset",[])),
 HF_status=globals().get("HF_STATUS","unknown"),
 HF_URL=globals().get("HF_URL"),
 gates=globals().get("release_gates",{}))
(OUT/"statistics_summary.json").write_text(json.dumps(summ,indent=2))
allowed={
 "selected_projections.json","baseline_metrics.json",
 "all28_bruteforce_trials.json","all28_cumulative_quality.json",
 "best_all28_trainable.json","all28_training_progress.json",
 "all28_training_summary.json","greedy_accepted_subset.json",
 "all28_test_comparison.json","statistics_summary.json",
 "huggingface_status.json","hf_upload_error.txt"
}
files=[OUT/n for n in sorted(allowed) if (OUT/n).is_file()]
for n in ("v55_report.json","provenance.json","README.md"):
    p=OUT/"standalone_model"/n
    if p.is_file():files.append(p)
assert files,"No statistics to download"
mem=io.StringIO()
writer=csv.writer(mem)
writer.writerow(("layer","width","steps","saved_parameters","gold","agree","JS","mse"))
for r in globals().get("all_trials",[]):
    q=r.get("single_layer_screen") or {}
    writer.writerow((r.get("layer"),r.get("width"),r.get("steps"),
       r.get("parameter_saving"),q.get("gold_accuracy"),
       q.get("teacher_agreement"),q.get("mean_js"),r.get("last_relative_mse")))
csvbytes=mem.getvalue().encode("utf-8")
manifest=dict(date_utc=datetime.now(timezone.utc).isoformat(),
 archive_scope="statistics-only; full model on HF iff all release gates pass",
 files=[])
with zipfile.ZipFile(ZIP,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in files:
        a=("model_metadata/" if p.parent!=OUT else "")+p.name
        data=p.read_bytes()
        z.writestr(a,data)
        manifest["files"].append(dict(path=a,sha256=hashlib.sha256(data).hexdigest(),
                                     bytes=len(data)))
    z.writestr("all28_trial_comparison.csv",csvbytes)
    manifest["files"].append(dict(path="all28_trial_comparison.csv",
                    sha256=hashlib.sha256(csvbytes).hexdigest(),bytes=len(csvbytes)))
    z.writestr("SHA256_manifest.json",json.dumps(manifest,indent=2))
with zipfile.ZipFile(ZIP) as z:
    banned=[x for x in z.namelist() if x.lower().endswith(
        (".safetensors",".bin",".pt",".pth","tokenizer.json"))]
    assert not banned, f"WEIGHTS IN STATISTICS ZIP: {banned}"
    assert z.testzip() is None
print("STATISTICS ARCHIVE:",ZIP,"bytes:",ZIP.stat().st_size)
print("HF MODEL:",globals().get("HF_URL") or "not published")
try:
    from IPython.display import display,FileLink
    display(FileLink(str(ZIP)))
except ImportError:
    pass
try:
    from google.colab import files as files_colab
    files_colab.download(str(ZIP))
    print("Automatic statistics ZIP download initiated")
except Exception as err:
    print("Auto-download blocked; ZIP available in /content:",repr(err))


STATISTICS ARCHIVE: /content/Laya_V55_All28_Statistics.zip bytes: 37685
HF MODEL: not published


/content/Laya_V55_All28_Statistics.zip

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Automatic statistics ZIP download initiated
